<a href="https://colab.research.google.com/github/RafaelSinhorini/PRECO_IMOVEIS_SP_TRATAMENTO_DADOS/blob/main/PRECO_IM%C3%93VEIS_SP_TRATAMENTO_DADOS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tratamento e Preparação dos Dados

## 1. Apresentação

Este notebook corresponde à segunda etapa do projeto de Ciência de Dados voltado à predição de preços de imóveis anunciados na cidade de São Paulo.

Após a análise exploratória realizada no notebook anterior, foram identificados valores ausentes, registros duplicados, inconsistências, valores extremos, formatos inadequados e variáveis com baixa relevância para o objetivo do projeto.

Nesta etapa, serão aplicadas as técnicas necessárias para limpar, transformar e preparar a base de dados para o desenvolvimento dos modelos de Machine Learning. Entre os principais procedimentos previstos estão:

- remoção de registros duplicados;
- tratamento de valores ausentes;
- correção de tipos e formatos das variáveis;
- tratamento de valores inconsistentes e extremos;
- exclusão de variáveis que não serão utilizadas;
- tratamento de categorias pouco frequentes;
- codificação das variáveis categóricas;
- preparação da base final para a etapa de modelagem.

As decisões tomadas durante o tratamento serão documentadas e justificadas, preservando a base original e garantindo a rastreabilidade das alterações realizadas.

## 2. Importação das bibliotecas

Nesta etapa, serão importadas as bibliotecas Python necessárias para o tratamento, a transformação e a preparação dos dados.

Outras bibliotecas poderão ser adicionadas posteriormente, conforme as necessidades identificadas durante o desenvolvimento do projeto.

In [ ]:
# Bibliotecas para manipulação e tratamento dos dados
import pandas as pd
import numpy as np

# Biblioteca para visualizações de apoio
import matplotlib.pyplot as plt

# Configuração para exibição dos valores numéricos
pd.options.display.float_format = '{:,.2f}'.format

## 3. Conexão com o Google Drive

Nesta etapa, o Google Drive será conectado ao ambiente do Google Colab, permitindo o acesso à pasta do projeto e à base de dados original utilizada no notebook de análise exploratória.
`

In [ ]:
# Conectar o Google Drive ao Google Colab
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 4. Carregamento e cópia da base de dados

Nesta etapa, será carregada a base de dados original utilizada na análise exploratória. Em seguida, será criada uma cópia independente para a aplicação dos tratamentos, preservando os dados originais sem alterações.

Todos os procedimentos de limpeza e transformação serão realizados na variável **df_tratado**. Ao final do processo, essa base será exportada como um novo arquivo CSV.

In [ ]:
# Caminho da base de dados original
caminho_base = '/content/drive/MyDrive/Colab Notebooks/PROJETO IV- UNIVESP/data_preco_sp.csv'

# Carregar a base original utilizando ponto e vírgula como separador
df_original = pd.read_csv(
    caminho_base,
    sep=';'
)

# Criar uma cópia independente para o tratamento
df_tratado = df_original.copy()

# Verificar as dimensões
print(f'Base original: {df_original.shape[0]} linhas e {df_original.shape[1]} colunas')
print(f'Cópia para tratamento: {df_tratado.shape[0]} linhas e {df_tratado.shape[1]} colunas')

Base original: 23433 linhas e 38 colunas
Cópia para tratamento: 23433 linhas e 38 colunas


### 4.1. Verificação inicial da cópia de trabalho

Antes de iniciar o tratamento, será verificado se a cópia de trabalho preserva as dimensões, os nomes das colunas e os tipos de dados da base original. Essa conferência garante que os dados foram carregados corretamente e estabelece um ponto de referência para acompanhar as alterações realizadas ao longo do notebook.

In [ ]:
# Verificar as dimensões da cópia
print(f'Dimensões da base: {df_tratado.shape}')

# Verificar os nomes e tipos das colunas
df_tratado.info()

Dimensões da base: (23433, 38)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23433 entries, 0 to 23432
Data columns (total 38 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   index                   23433 non-null  int64  
 1   title                   23433 non-null  object 
 2   price                   23433 non-null  int64  
 3   oldPrice                890 non-null    float64
 4   location                23433 non-null  object 
 5   date                    23433 non-null  object 
 6   destaque                23433 non-null  bool   
 7   Categoria               23433 non-null  object 
 8   Tipo                    23433 non-null  object 
 9   Condomínio              23266 non-null  float64
 10  Área útil               23433 non-null  int64  
 11  Quartos                 23433 non-null  object 
 12  Banheiros               23427 non-null  object 
 13  Vagas na garagem        23433 non-null  object 
 14  Detalhe

In [ ]:
# Restaurar a cópia de trabalho a partir da base original
df_tratado = df_original.copy()

# Confirmar a restauração
print(f'Base restaurada: {df_tratado.shape[0]} linhas e {df_tratado.shape[1]} colunas')

Base restaurada: 23433 linhas e 38 colunas


## 5. Tratamento dos registros duplicados

A análise exploratória identificou a presença de registros duplicados na base de dados. Essas repetições podem distorcer as análises estatísticas e influenciar o treinamento do modelo, aumentando artificialmente a representatividade de determinados anúncios.

Nesta etapa, serão removidas somente as ocorrências excedentes, mantendo a primeira ocorrência de cada registro duplicado. Esse tratamento será realizado antes da exclusão de qualquer variável, garantindo que a identificação considere todas as informações originalmente disponíveis.

A remoção será aplicada exclusivamente à cópia **df_tratado**, preservando integralmente a base de dados original.

In [ ]:
# Quantidade de registros antes do tratamento
registros_antes = len(df_tratado)

# Identificar registros duplicados desconsiderando apenas a coluna index
colunas_comparacao = df_tratado.columns.drop('index')
duplicados_identificados = df_tratado.duplicated(
    subset=colunas_comparacao
).sum()

# Remover as repetições, mantendo a primeira ocorrência
df_tratado.drop_duplicates(
    subset=colunas_comparacao,
    keep='first',
    inplace=True
)

# Reorganizar o índice do DataFrame
df_tratado.reset_index(drop=True, inplace=True)

# Exibir os resultados
print(f'Registros antes do tratamento: {registros_antes}')
print(f'Registros duplicados removidos: {duplicados_identificados}')
print(f'Registros após o tratamento: {len(df_tratado)}')
print(
    f'Duplicados restantes: '
    f'{df_tratado.duplicated(subset=colunas_comparacao).sum()}'
)

Registros antes do tratamento: 23433
Registros duplicados removidos: 1784
Registros após o tratamento: 21649
Duplicados restantes: 0


## 6. Tratamento das variáveis não selecionadas

Nesta etapa, serão removidas da cópia de trabalho as variáveis que não serão utilizadas no futuro modelo de predição.

A remoção considera a ausência de relação direta com o objetivo do projeto, a presença de textos não estruturados, a baixa disponibilidade de informações ou a redundância com outras informações da base.

As variáveis inicialmente selecionadas para remoção são:

- **index:** identifica numericamente os registros;
- **title:** contém textos livres dos anúncios;
- **destaque:** representa uma característica comercial da publicação;
- **Detalhes do imóvel:** contém informações textuais não estruturadas;
- **Detalhes do condomínio:** contém informações textuais não estruturadas;
- **oldPrice:** apresenta elevado percentual de valores ausentes;
- - **date:** representa um período curto de coleta e não oferece informação temporal relevante para o modelo.

A remoção será aplicada exclusivamente à cópia **df_tratado**, preservando integralmente a base de dados original.

In [ ]:
# Variáveis selecionadas para remoção
colunas_remover = [
    'index',
    'title',
    'destaque',
    'Detalhes do imóvel',
    'Detalhes do condomínio',
    'oldPrice',
    'date'
]

# Registrar as dimensões antes da remoção
dimensoes_antes = df_tratado.shape

# Remover as variáveis da cópia de trabalho
df_tratado.drop(columns=colunas_remover, inplace=True)

# Exibir os resultados
print(f'Dimensões antes da remoção: {dimensoes_antes}')
print(f'Dimensões após a remoção: {df_tratado.shape}')
print(f'Quantidade de colunas removidas: {len(colunas_remover)}')

Dimensões antes da remoção: (21649, 38)
Dimensões após a remoção: (21649, 31)
Quantidade de colunas removidas: 7


## 7. Correção dos tipos de dados

As variáveis **Quartos**, **Banheiros** e **Vagas na garagem** estão armazenadas como texto devido à presença da categoria **“5 ou mais”**. Esse formato impede sua utilização direta em cálculos estatísticos e algoritmos de Machine Learning.

Nesta etapa, a categoria **“5 ou mais”** será convertida para o valor numérico **5**, representando o limite mínimo conhecido para esses registros. Em seguida, as três variáveis serão convertidas para formato numérico.

Os valores ausentes da variável **Banheiros** serão preservados temporariamente e tratados em uma etapa posterior.

In [ ]:
# Substituir a categoria textual pelo menor valor conhecido
colunas_converter = ['Quartos', 'Banheiros', 'Vagas na garagem']

for coluna in colunas_converter:
    df_tratado[coluna] = df_tratado[coluna].replace('5 ou mais', 5)
    df_tratado[coluna] = pd.to_numeric(
        df_tratado[coluna],
        errors='coerce'
    )

# Verificar os tipos e valores encontrados após a conversão
print(df_tratado[colunas_converter].dtypes)

for coluna in colunas_converter:
    print(f'\nValores da variável {coluna}:')
    print(sorted(df_tratado[coluna].dropna().unique()))

Quartos               int64
Banheiros           float64
Vagas na garagem      int64
dtype: object

Valores da variável Quartos:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Valores da variável Banheiros:
[np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]

Valores da variável Vagas na garagem:
[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


## 8. Tratamento dos valores ausentes

Nesta etapa, serão identificados os valores ausentes que permaneceram na base após a remoção das variáveis não selecionadas e dos registros duplicados.

Cada variável será avaliada individualmente, considerando sua importância para o modelo, a quantidade de dados faltantes e as características de sua distribuição. O objetivo é definir estratégias de tratamento que preservem a qualidade e a representatividade dos dados.

In [ ]:
# Calcular a quantidade e o percentual de valores ausentes
valores_ausentes = pd.DataFrame({
    'Quantidade': df_tratado.isnull().sum(),
    'Percentual (%)': (
        df_tratado.isnull().mean() * 100
    ).round(2)
})

# Exibir somente as variáveis com valores ausentes
valores_ausentes = valores_ausentes[
    valores_ausentes['Quantidade'] > 0
].sort_values(
    by='Quantidade',
    ascending=False
)

valores_ausentes

,Quantidade,Percentual (%)
IPTU,201,0.93
Condomínio,167,0.77
Banheiros,6,0.03


### 8.1. Remoção da variável IPTU

A variável **IPTU** foi mantida nas etapas iniciais para que sua qualidade e disponibilidade fossem avaliadas em conjunto com as demais variáveis da base. Após a remoção dos registros duplicados, verificou-se que ainda existem **201 valores ausentes**, correspondentes a **0,93% dos registros**.

Além das ausências, a análise exploratória anterior identificou valores simbólicos e extremamente elevados, indicando possíveis inconsistências de preenchimento. Também foi considerado que o IPTU não está previsto como uma informação obrigatória de entrada no futuro sistema de predição.

Diante desses fatores, optou-se por remover a variável **IPTU** nesta etapa. A decisão foi tomada somente após sua análise individual e a confirmação dos valores ausentes remanescentes, permitindo que a exclusão fosse fundamentada em evidências, e não realizada antecipadamente.


In [ ]:
# Remover a variável IPTU da cópia de trabalho
df_tratado.drop(columns=['IPTU'], inplace=True)

# Confirmar a remoção
print(f'A variável IPTU permanece na base? {"IPTU" in df_tratado.columns}')
print(f'Dimensões atuais da base: {df_tratado.shape}')

A variável IPTU permanece na base? False
Dimensões atuais da base: (21649, 30)


### 8.2. Tratamento dos valores ausentes da variável Condomínio

A variável **Condomínio** apresenta **167 valores ausentes**, correspondentes a **0,77% dos registros** da base tratada. Antes de definir uma estratégia de preenchimento, é necessário investigar as características dos imóveis associados a essas ausências.

Para isso, os registros serão analisados conforme as variáveis **Categoria**, **Tipo** e **ZONA**. Essa investigação permitirá verificar se os valores ausentes estão concentrados em grupos específicos, como casas que possivelmente não possuem cobrança de condomínio, ou se representam informações não preenchidas nos anúncios.

A decisão de tratamento será tomada somente após essa avaliação, evitando o preenchimento generalizado com valores que possam não representar corretamente cada imóvel.

In [ ]:
# Selecionar registros com Condomínio ausente
condominio_ausente = df_tratado[
    df_tratado['Condomínio'].isna()
]

print(f'Total de registros analisados: {len(condominio_ausente)}')

print('\nDistribuição por Categoria:')
print(condominio_ausente['Categoria'].value_counts())

print('\nDistribuição por Tipo:')
print(condominio_ausente['Tipo'].value_counts())

print('\nDistribuição por Zona:')
print(condominio_ausente['ZONA'].value_counts())

Total de registros analisados: 167

Distribuição por Categoria:
Categoria
Casas           138
Apartamentos     29
Name: count, dtype: int64

Distribuição por Tipo:
Tipo
Padrão                         140
Casa de vila                    18
Duplex ou triplex                2
Venda - casa em rua pública      2
Casa de condomínio               2
Kitnet                           1
Venda - casa em vila             1
Loft                             1
Name: count, dtype: int64

Distribuição por Zona:
ZONA
LESTE     90
NORTE     25
SUL       25
OESTE     24
CENTRO     3
Name: count, dtype: int64


In [ ]:
# Estatísticas do condomínio por categoria e tipo
analise_condominio_grupos = (
    df_tratado[df_tratado['Condomínio'].notna()]
    .groupby(['Categoria', 'Tipo'])['Condomínio']
    .agg(
        Quantidade='count',
        Mediana='median',
        Media='mean',
        Valores_zero=lambda x: (x == 0).sum(),
        Valores_um=lambda x: (x == 1).sum()
    )
    .sort_values(by='Quantidade', ascending=False)
)

analise_condominio_grupos

Quantidade  Mediana     Media  Valores_zero  \
Categoria    Tipo                                                              
Apartamentos Padrão                   15614   976.50  2,114.77           886   
Casas        Padrão                    4407     0.00     32.86          3926   
Apartamentos Cobertura                  546 2,300.00 29,442.52            16   
Casas        Casa de condomínio         493   200.00  1,614.34            83   
Apartamentos Loft                       252   753.00  1,089.76            26   
             Kitnet                     106   400.00    377.58            11   
Casas        Casa de vila                48     0.00     96.48            39   
Apartamentos Duplex ou triplex           16 1,325.00  2,077.69             1   

                                 Valores_um  
Categoria    Tipo                            
Apartamentos Padrão                     211  
Casas        Padrão                     308  
Apartamentos Cobertura                    4  
Casas        Casa de condomínio          15  
Apartamentos Loft                         3  
             Kitnet                       3  
Casas        Casa de vila                 1  
Apartamentos Duplex ou triplex            0

#### Estratégia adotada para o preenchimento

A investigação demonstrou que os valores de **Condomínio** variam significativamente conforme a **Categoria** e o **Tipo** do imóvel. Casas do tipo Padrão e Casas de vila apresentaram mediana igual a **R$ 0,00**, enquanto Apartamentos, Coberturas, Lofts e Casas de condomínio apresentaram medianas diferentes e superiores.

Diante desse comportamento, não seria adequado preencher todos os valores ausentes com zero ou utilizar uma única mediana para toda a base. Essas estratégias poderiam atribuir valores incompatíveis com as características dos imóveis.

Assim, os valores ausentes serão preenchidos pela **mediana do grupo formado pelas variáveis Categoria e Tipo**. Dessa forma, cada registro receberá um valor estimado a partir de imóveis com características semelhantes, preservando melhor as diferenças existentes entre os grupos.

Antes do preenchimento definitivo, será verificado se todos os registros com valor ausente pertencem a grupos que possuem uma mediana disponível.

#### Validação das medianas por grupo

Antes de preencher os valores ausentes, será verificado se todas as combinações de **Categoria** e **Tipo** possuem uma mediana de Condomínio disponível.

Essa validação é necessária porque alguns grupos podem ser formados exclusivamente por registros com valores ausentes. Nesses casos, não será possível utilizar a mediana específica do grupo, sendo necessária uma estratégia complementar de preenchimento.


In [ ]:
# Calcular a mediana do Condomínio por Categoria e Tipo
medianas_condominio = df_tratado.groupby(
    ['Categoria', 'Tipo']
)['Condomínio'].transform('median')

# Verificar registros ausentes que também não possuem mediana no grupo
sem_mediana_grupo = df_tratado[
    df_tratado['Condomínio'].isna() &
    medianas_condominio.isna()
]

print(f'Registros ausentes sem mediana disponível no grupo: {len(sem_mediana_grupo)}')

Registros ausentes sem mediana disponível no grupo: 3


In [ ]:
# Exibir os registros sem mediana disponível no grupo
sem_mediana_grupo[
    ['Categoria', 'Tipo', 'Condomínio', 'location', 'ZONA']
]

,Categoria,Tipo,Condomínio,location,ZONA
919,Casas,Venda - casa em rua pública,NaN,"São Paulo, Jardim Umarizal",SUL
17726,Casas,Venda - casa em vila,NaN,"São Paulo, Itaquera",LESTE
17731,Casas,Venda - casa em rua pública,NaN,"São Paulo, Jardim Nossa Senhora do Carmo",LESTE


#### Conclusão da investigação e estratégia de preenchimento

A análise dos **167 registros** com valores ausentes na variável **Condomínio** demonstrou que as ausências estão distribuídas entre diferentes categorias e tipos de imóveis. Também foi identificado que os valores típicos de condomínio variam significativamente entre esses grupos.

Por esse motivo, não será utilizado um único valor para toda a base. A estratégia de tratamento será:

- preencher os valores ausentes utilizando a **mediana do grupo formado pelas variáveis Categoria e Tipo**;
- para os **3 registros** cujos grupos não possuem mediana disponível, utilizar o valor **R$ 0,00**, pois correspondem a casas anunciadas em rua pública ou vila e não estão classificadas como casas de condomínio;
- preservar a variável **Categoria**, que será utilizada apenas como referência para o agrupamento nesta etapa;
- verificar, após o preenchimento, se ainda existem valores ausentes na variável **Condomínio**.

Essa estratégia busca preservar as diferenças entre os tipos de imóveis e evitar a atribuição de um mesmo valor de condomínio a registros com características distintas.

In [ ]:
# Calcular a mediana do Condomínio por Categoria e Tipo
medianas_condominio = df_tratado.groupby(
    ['Categoria', 'Tipo']
)['Condomínio'].transform('median')

# Preencher pela mediana do respectivo grupo
df_tratado['Condomínio'] = (
    df_tratado['Condomínio']
    .fillna(medianas_condominio)
)

# Preencher com zero os 3 registros que não possuíam mediana no grupo
df_tratado['Condomínio'] = (
    df_tratado['Condomínio']
    .fillna(0)
)

# Validar o tratamento
print(
    f'Valores ausentes em Condomínio após o tratamento: '
    f'{df_tratado["Condomínio"].isna().sum()}'
)

print(
    f'Quantidade de registros com Condomínio igual a zero: '
    f'{(df_tratado["Condomínio"] == 0).sum()}'
)

Valores ausentes em Condomínio após o tratamento: 0
Quantidade de registros com Condomínio igual a zero: 5124


### 8.3. Tratamento dos valores ausentes da variável Banheiros

A variável **Banheiros** apresenta apenas **6 valores ausentes**, correspondentes a aproximadamente **0,03% dos registros** da base tratada.

Antes de realizar o preenchimento, será investigado o perfil desses imóveis por meio de características como **Categoria**, **Tipo**, **Quartos** e **Área útil**. Essa análise permitirá identificar uma estratégia de preenchimento compatível com imóveis semelhantes, evitando a atribuição arbitrária de valores.

Após o tratamento das ausências, a variável **Banheiros** será convertida do tipo `float64` para `int64`, pois passará a conter somente valores numéricos inteiros.

In [ ]:
# Selecionar registros com valores ausentes em Banheiros
banheiros_ausentes = df_tratado[
    df_tratado['Banheiros'].isna()
]

# Exibir características relevantes desses imóveis
banheiros_ausentes[
    [
        'Categoria',
        'Tipo',
        'Quartos',
        'Área útil',
        'Vagas na garagem',
        'location',
        'ZONA'
    ]
]

,Categoria,Tipo,Quartos,Área útil,Vagas na garagem,location,ZONA
331,Apartamentos,Padrão,2,50,0,"São Paulo, Brás",LESTE
4088,Apartamentos,Kitnet,1,29,0,"São Paulo, Bela Vista",CENTRO
7403,Apartamentos,Padrão,2,46,0,"São Paulo, São Miguel Paulista",LESTE
9448,Casas,Padrão,5,250,5,"São Paulo, Cidade Patriarca",LESTE
11598,Casas,Padrão,0,0,0,"São Paulo, Parque do Morumbi",SUL
14316,Casas,Padrão,2,50,0,"São Paulo, Parque Tietê",NORTE


#### Conclusão da investigação dos valores ausentes em Banheiros

A investigação dos **6 registros** com valores ausentes na variável **Banheiros** identificou imóveis com características distintas de categoria, tipo, quantidade de quartos e área útil. Por esse motivo, não será adequado preencher todas as ausências com um único valor geral.

Um dos registros também apresenta **zero quartos e área útil igual a zero**, indicando inconsistências adicionais. Esse registro não será preenchido nesta etapa, pois deverá ser reavaliado posteriormente durante o tratamento dos valores inválidos.

Para os demais registros, será calculada a **mediana da quantidade de banheiros em grupos de imóveis semelhantes**, considerando principalmente as variáveis **Categoria**, **Tipo** e **Quartos**. Após o preenchimento, será verificado se ainda existem valores ausentes e a variável será convertida para o tipo numérico inteiro.

In [ ]:
# Calcular a mediana de Banheiros por grupos semelhantes
medianas_banheiros = df_tratado.groupby(
    ['Categoria', 'Tipo', 'Quartos']
)['Banheiros'].transform('median')

# Visualizar os valores sugeridos para os registros ausentes
analise_preenchimento_banheiros = df_tratado.loc[
    df_tratado['Banheiros'].isna(),
    ['Categoria', 'Tipo', 'Quartos', 'Área útil', 'Banheiros']
].copy()

analise_preenchimento_banheiros['Mediana sugerida'] = medianas_banheiros[
    df_tratado['Banheiros'].isna()
]

analise_preenchimento_banheiros

,Categoria,Tipo,Quartos,Área útil,Banheiros,Mediana sugerida
331,Apartamentos,Padrão,2,50,NaN,1.00
4088,Apartamentos,Kitnet,1,29,NaN,1.00
7403,Apartamentos,Padrão,2,46,NaN,1.00
9448,Casas,Padrão,5,250,NaN,4.00
11598,Casas,Padrão,0,0,NaN,3.00
14316,Casas,Padrão,2,50,NaN,2.00


#### Avaliação das medianas sugeridas para Banheiros

A análise por grupos de imóveis semelhantes apresentou valores coerentes para **5 dos 6 registros** com quantidade de banheiros ausente. As medianas sugeridas variaram entre **1 e 4 banheiros**, conforme a Categoria, o Tipo e a quantidade de Quartos de cada imóvel.

Entretanto, um dos registros apresenta simultaneamente **zero quartos**, **área útil igual a zero** e ausência na variável **Banheiros**. Embora a mediana do grupo tenha sugerido 3 banheiros, esse preenchimento não será realizado neste momento, pois as características do registro indicam possíveis inconsistências adicionais.

Antes de concluir o tratamento da variável **Banheiros**, serão investigados todos os registros com quantidade de quartos igual a zero. A análise considerará a Categoria, o Tipo, a Área útil e o preço dos imóveis, permitindo verificar se esses registros representam erros de preenchimento, imóveis comerciais ou outras situações incompatíveis com o objetivo do projeto.

In [ ]:
# Selecionar registros com quantidade de quartos igual a zero
quartos_zero = df_tratado[
    df_tratado['Quartos'] == 0
]

print(f'Total de registros com zero quartos: {len(quartos_zero)}')

# Exibir informações relevantes desses registros
quartos_zero[
    [
        'Categoria',
        'Tipo',
        'price',
        'Área útil',
        'Quartos',
        'Banheiros',
        'Vagas na garagem',
        'location',
        'ZONA'
    ]
]

Total de registros com zero quartos: 24


,Categoria,Tipo,price,Área útil,Quartos,Banheiros,Vagas na garagem,location,ZONA
348,Apartamentos,Padrão,5000000,1861,0,5.00,5,"São Paulo, Quarta Parada",LESTE
350,Apartamentos,Kitnet,317000,30,0,1.00,1,"São Paulo, Vila Clementino",SUL
717,Apartamentos,Kitnet,178000,27,0,1.00,0,"São Paulo, Campos Elíseos",CENTRO
720,Apartamentos,Padrão,345000,12,0,1.00,0,"São Paulo, Vila Buarque",CENTRO
1073,Apartamentos,Padrão,195000,16,0,1.00,0,"São Paulo, Cidade Antônio Estevão de Carvalho",LESTE
1422,Apartamentos,Padrão,1300000,340,0,4.00,0,"São Paulo, Vila Guaca",NORTE
1776,Casas,Padrão,450000,132,0,1.00,2,"São Paulo, Vila Ema",LESTE
1806,Casas,Padrão,1350000,150,0,1.00,3,"São Paulo, Alto da Mooca",LESTE
2252,Casas,Padrão,1700000,467,0,3.00,5,"São Paulo, Quarta Parada",LESTE
2827,Apartamentos,Padrão,5800000,2662,0,5.00,5,"São Paulo, Imirim",NORTE


#### Tratamento dos registros com zero quartos

A investigação identificou **24 registros** com quantidade de quartos igual a zero. Nos imóveis classificados como **Kitnet**, **Loft** ou apartamentos de pequena metragem, esse valor pode representar uma característica válida, indicando a existência de um ambiente integrado, sem divisão exclusiva para dormitório. Por esse motivo, esses registros serão mantidos.

Em contrapartida, foi identificado um registro classificado como **Casa** que apresenta simultaneamente preço de **R$ 500,00**, área útil igual a **0 m²**, zero quartos e quantidade de banheiros ausente. A combinação dessas características demonstra que o registro não contém informações suficientes ou coerentes para representar um imóvel válido para o objetivo do projeto.

Dessa forma, apenas esse registro será removido da base tratada. Os outros registros com zero quartos serão preservados neste momento e poderão ser avaliados novamente durante o tratamento das inconsistências.

In [ ]:
# Identificar o registro claramente inválido
condicao_registro_invalido = (
    (df_tratado['price'] == 500) &
    (df_tratado['Área útil'] == 0) &
    (df_tratado['Quartos'] == 0) &
    (df_tratado['Banheiros'].isna()) &
    (df_tratado['Categoria'] == 'Casas')
)

# Contar registros antes da remoção
registros_antes = len(df_tratado)
registros_identificados = condicao_registro_invalido.sum()

# Remover somente o registro identificado
df_tratado = df_tratado[
    ~condicao_registro_invalido
].copy()

# Reorganizar o índice
df_tratado.reset_index(drop=True, inplace=True)

# Validar o tratamento
print(f'Registros inválidos identificados: {registros_identificados}')
print(f'Registros antes da remoção: {registros_antes}')
print(f'Registros após a remoção: {len(df_tratado)}')

Registros inválidos identificados: 1
Registros antes da remoção: 21649
Registros após a remoção: 21648


#### Preenchimento dos valores ausentes em Banheiros

Após a remoção do registro inválido, permaneceram **5 valores ausentes** na variável **Banheiros**. Esses valores serão preenchidos pelas medianas dos respectivos grupos, formados pelas variáveis **Categoria**, **Tipo** e **Quartos**.

Após o preenchimento, será verificada a inexistência de valores ausentes e a variável será convertida para o tipo inteiro (`int64`).

In [ ]:
# Recalcular a mediana de Banheiros por grupos semelhantes
medianas_banheiros = df_tratado.groupby(
    ['Categoria', 'Tipo', 'Quartos']
)['Banheiros'].transform('median')

# Preencher os valores ausentes com a mediana do respectivo grupo
df_tratado['Banheiros'] = (
    df_tratado['Banheiros']
    .fillna(medianas_banheiros)
)

# Converter a variável para o tipo inteiro
df_tratado['Banheiros'] = df_tratado['Banheiros'].astype('int64')

# Validar o tratamento
print(
    f'Valores ausentes em Banheiros: '
    f'{df_tratado["Banheiros"].isna().sum()}'
)

print(
    f'Tipo da variável Banheiros: '
    f'{df_tratado["Banheiros"].dtype}'
)

print(
    f'Valores encontrados: '
    f'{sorted(df_tratado["Banheiros"].unique())}'
)

Valores ausentes em Banheiros: 0
Tipo da variável Banheiros: int64
Valores encontrados: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


### 8.4. Tratamento dos registros com zero quartos

Após o tratamento dos valores ausentes em **Banheiros**, será retomada a análise dos registros que apresentam quantidade de quartos igual a zero.

Os imóveis classificados como **Kitnet** ou **Loft** serão desconsiderados inicialmente nesta investigação, pois podem possuir ambientes integrados e não apresentar uma divisão exclusiva destinada ao quarto.

Também serão preservados os apartamentos com **área útil inferior a 25 m²**, pois podem apresentar características semelhantes às de estúdios ou imóveis compactos, mesmo que estejam classificados como do tipo **Padrão**.

Para os demais imóveis, especialmente casas e apartamentos com áreas maiores, a quantidade igual a zero pode indicar ausência de informação ou erro de preenchimento. Esses registros serão analisados antes da definição de qualquer estratégia de correção ou remoção.

In [ ]:
# Selecionar registros com zero quartos que ainda exigem investigação
quartos_zero_investigar = df_tratado[
    (df_tratado['Quartos'] == 0) &
    (~df_tratado['Tipo'].isin(['Kitnet', 'Loft'])) &
    ~(
        (df_tratado['Categoria'] == 'Apartamentos') &
        (df_tratado['Área útil'] < 25)
    )
]

print(
    f'Registros com zero quartos que ainda exigem investigação: '
    f'{len(quartos_zero_investigar)}'
)

# Exibir as características relevantes
quartos_zero_investigar[
    [
        'Categoria',
        'Tipo',
        'price',
        'Área útil',
        'Banheiros',
        'Vagas na garagem',
        'location',
        'ZONA'
    ]
]


Registros com zero quartos que ainda exigem investigação: 17


,Categoria,Tipo,price,Área útil,Banheiros,Vagas na garagem,location,ZONA
348,Apartamentos,Padrão,5000000,1861,5,5,"São Paulo, Quarta Parada",LESTE
1422,Apartamentos,Padrão,1300000,340,4,0,"São Paulo, Vila Guaca",NORTE
1776,Casas,Padrão,450000,132,1,2,"São Paulo, Vila Ema",LESTE
1806,Casas,Padrão,1350000,150,1,3,"São Paulo, Alto da Mooca",LESTE
2252,Casas,Padrão,1700000,467,3,5,"São Paulo, Quarta Parada",LESTE
2827,Apartamentos,Padrão,5800000,2662,5,5,"São Paulo, Imirim",NORTE
3256,Casas,Padrão,270000,75,1,0,"São Paulo, Jardim Casa Grande",SUL
5137,Casas,Padrão,1100000,160,4,3,"São Paulo, Ipiranga",SUL
5437,Casas,Padrão,485000,133,4,0,"São Paulo, Vila Bandeirantes",NORTE
5584,Casas,Padrão,980000,193,4,2,"São Paulo, Mirandópolis",SUL


#### Estratégia para o tratamento dos registros com zero quartos

Após desconsiderar **Kitnets**, **Lofts** e apartamentos com área útil inferior a **25 m²**, permaneceram **17 registros** com quantidade de quartos igual a zero que exigem investigação.

Para esses registros, será calculada a mediana de quartos de imóveis semelhantes, agrupados pelas variáveis **Categoria**, **Tipo** e **Banheiros**. Essas características permitem comparar imóveis com estrutura semelhante e estimar um valor mais representativo para a quantidade de quartos.

A variável **Área útil** não será utilizada inicialmente nesse agrupamento, pois alguns dos registros analisados apresentam metragens potencialmente inconsistentes. A utilização desses valores poderia prejudicar a formação dos grupos e gerar estimativas inadequadas.

Inicialmente, os valores sugeridos serão apenas apresentados para avaliação. A base não será alterada até que as medianas calculadas sejam consideradas coerentes para o tratamento.

In [ ]:
# Calcular a mediana de quartos por grupos semelhantes
medianas_quartos = df_tratado.groupby(
    ['Categoria', 'Tipo', 'Banheiros']
)['Quartos'].transform('median')

# Criar tabela com os 17 registros que exigem investigação
analise_preenchimento_quartos = quartos_zero_investigar[
    [
        'Categoria',
        'Tipo',
        'Banheiros',
        'Área útil',
        'Quartos'
    ]
].copy()

# Adicionar a mediana sugerida
analise_preenchimento_quartos['Mediana sugerida'] = medianas_quartos.loc[
    quartos_zero_investigar.index
]

analise_preenchimento_quartos

,Categoria,Tipo,Banheiros,Área útil,Quartos,Mediana sugerida
348,Apartamentos,Padrão,5,1861,0,4.00
1422,Apartamentos,Padrão,4,340,0,3.00
1776,Casas,Padrão,1,132,0,2.00
1806,Casas,Padrão,1,150,0,2.00
2252,Casas,Padrão,3,467,0,3.00
2827,Apartamentos,Padrão,5,2662,0,4.00
3256,Casas,Padrão,1,75,0,2.00
5137,Casas,Padrão,4,160,0,3.00
5437,Casas,Padrão,4,133,0,3.00
5584,Casas,Padrão,4,193,0,3.00


#### Aplicação do tratamento na variável Quartos

As medianas sugeridas foram consideradas coerentes para os **17 registros** investigados. Dessa forma, os valores iguais a zero desses imóveis serão substituídos pela mediana de quartos do respectivo grupo, formado pelas variáveis **Categoria**, **Tipo** e **Banheiros**.

Os valores iguais a zero considerados plausíveis em **Kitnets**, **Lofts** e apartamentos com área útil inferior a **25 m²** serão preservados.

In [ ]:
# Recalcular a mediana de quartos por grupos semelhantes
medianas_quartos = df_tratado.groupby(
    ['Categoria', 'Tipo', 'Banheiros']
)['Quartos'].transform('median')

# Identificar novamente os registros que devem ser tratados
condicao_tratamento_quartos = (
    (df_tratado['Quartos'] == 0) &
    (~df_tratado['Tipo'].isin(['Kitnet', 'Loft'])) &
    ~(
        (df_tratado['Categoria'] == 'Apartamentos') &
        (df_tratado['Área útil'] < 25)
    )
)

# Quantidade de registros que serão alterados
registros_tratados = condicao_tratamento_quartos.sum()

# Substituir os valores iguais a zero pelas medianas dos grupos
df_tratado.loc[
    condicao_tratamento_quartos,
    'Quartos'
] = medianas_quartos.loc[condicao_tratamento_quartos].astype('int64')

# Validar o tratamento
print(f'Registros corrigidos: {registros_tratados}')
print(
    f'Registros com zero quartos preservados: '
    f'{(df_tratado["Quartos"] == 0).sum()}'
)
print(f'Tipo da variável Quartos: {df_tratado["Quartos"].dtype}')

Registros corrigidos: 17
Registros com zero quartos preservados: 6
Tipo da variável Quartos: int64


In [ ]:
# Verificar a quantidade total de valores ausentes
total_ausentes = df_tratado.isna().sum().sum()

# Verificar valores ausentes por variável
ausentes_por_variavel = df_tratado.isna().sum()
ausentes_por_variavel = ausentes_por_variavel[
    ausentes_por_variavel > 0
]

print(f'Total de valores ausentes na base: {total_ausentes}')

if total_ausentes == 0:
    print('Não existem valores ausentes na base tratada.')
else:
    print('\nValores ausentes por variável:')
    print(ausentes_por_variavel)

Total de valores ausentes na base: 0
Não existem valores ausentes na base tratada.


## 9. Tratamento dos valores inconsistentes e extremos

Após o tratamento dos valores ausentes, serão investigados os valores inconsistentes e extremos presentes nas variáveis numéricas da base.

A análise será realizada individualmente, considerando critérios estatísticos e as características do mercado imobiliário. Valores extremos não serão removidos automaticamente, pois podem representar imóveis reais de alto padrão. A modificação ou exclusão ocorrerá somente quando houver evidências de erro, valor simbólico, problema de escala ou incompatibilidade entre as características do registro.

### 9.1. Investigação da variável Condomínio

Embora os valores ausentes da variável **Condomínio** já tenham sido tratados, ainda é necessário avaliar a consistência dos valores existentes.

Nesta etapa, serão investigados:

- valores negativos ou iguais a zero;
- valores simbólicos, especialmente **R$ 1,00**;
- valores muito elevados;
- possíveis erros de preenchimento ou escala;
- diferenças da distribuição conforme a Categoria e o Tipo do imóvel.

Inicialmente, será realizada apenas uma análise descritiva. Nenhum valor será alterado antes da definição e justificativa dos critérios de tratamento.

In [ ]:
# Estatísticas descritivas da variável Condomínio
print('Estatísticas descritivas:')
print(df_tratado['Condomínio'].describe())

# Contagem de valores específicos
print('\nValores negativos:')
print((df_tratado['Condomínio'] < 0).sum())

print('\nValores iguais a zero:')
print((df_tratado['Condomínio'] == 0).sum())

print('\nValores iguais a R$ 1,00:')
print((df_tratado['Condomínio'] == 1).sum())

# Percentis superiores para análise dos valores extremos
print('\nPercentis da variável Condomínio:')
print(
    df_tratado['Condomínio'].quantile(
        [0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999, 1.00]
    )
)

# Exibir os 20 maiores valores de condomínio
print('\n20 maiores valores de Condomínio:')
print(
    df_tratado.nlargest(
        20,
        'Condomínio'
    )[
        [
            'Condomínio',
            'price',
            'Categoria',
            'Tipo',
            'Área útil',
            'bairro',
            'ZONA'
        ]
    ]
)

Estatísticas descritivas:
count       21,648.00
mean         2,328.96
std         86,406.33
min              0.00
25%              1.00
50%            710.00
75%          1,470.00
max     11,800,000.00
Name: Condomínio, dtype: float64

Valores negativos:
0

Valores iguais a zero:
5123

Valores iguais a R$ 1,00:
545

Percentis da variável Condomínio:
0.50          710.00
0.75        1,470.00
0.90        2,631.00
0.95        3,600.00
0.99        6,000.00
0.99        7,500.00
1.00       28,222.89
1.00   11,800,000.00
Name: Condomínio, dtype: float64

20 maiores valores de Condomínio:
         Condomínio     price     Categoria                Tipo  Área útil  \
12233 11,800,000.00  18000000  Apartamentos           Cobertura       1181   
16781  2,800,000.00   1299710  Apartamentos           Cobertura        242   
9921   1,960,000.00    730000  Apartamentos              Padrão        163   
8937   1,775,338.00    999240  Apartamentos              Padrão         70   
5727   1,200,000.00   

#### Investigação dos valores de Condomínio iguais a R$ 1,00

A análise descritiva identificou **545 registros** com valor de Condomínio igual a **R$ 1,00**. Embora esse valor seja numericamente válido, ele provavelmente representa um preenchimento simbólico utilizado quando o valor real não foi informado no anúncio.

Antes de realizar qualquer substituição, esses registros serão analisados conforme a **Categoria** e o **Tipo** do imóvel. Essa investigação permitirá distinguir imóveis que possivelmente não possuem cobrança de condomínio daqueles em que o valor está ausente ou foi registrado simbolicamente.

Nenhum valor será alterado nesta etapa. A estratégia de tratamento será definida somente após a análise da distribuição desses registros.

In [ ]:
# Selecionar registros com Condomínio igual a R$ 1,00
condominio_um = df_tratado[
    df_tratado['Condomínio'] == 1
]

print(f'Total de registros analisados: {len(condominio_um)}')

print('\nDistribuição por Categoria:')
print(condominio_um['Categoria'].value_counts())

print('\nDistribuição por Categoria e Tipo:')
print(
    condominio_um.groupby(
        ['Categoria', 'Tipo']
    ).size().sort_values(ascending=False)
)

print('\nDistribuição por Zona:')
print(condominio_um['ZONA'].value_counts())

Total de registros analisados: 545

Distribuição por Categoria:
Categoria
Casas           324
Apartamentos    221
Name: count, dtype: int64

Distribuição por Categoria e Tipo:
Categoria     Tipo              
Casas         Padrão                308
Apartamentos  Padrão                211
Casas         Casa de condomínio     15
Apartamentos  Cobertura               4
              Kitnet                  3
              Loft                    3
Casas         Casa de vila            1
dtype: int64

Distribuição por Zona:
ZONA
LESTE     188
NORTE     186
OESTE      69
CENTRO     56
SUL        46
Name: count, dtype: int64


#### Estratégia para o tratamento dos valores iguais a R$ 1,00

A investigação identificou **545 registros** com valor de Condomínio igual a **R$ 1,00**, distribuídos entre **324 Casas** e **221 Apartamentos**.

A presença desse valor em diferentes tipos de imóveis, incluindo Apartamentos Padrão, Coberturas, Kitnets, Lofts e Casas de condomínio, indica que **R$ 1,00 provavelmente representa um valor simbólico**, utilizado quando a informação real não foi fornecida no anúncio.

Entretanto, o tratamento não será igual para todos os registros:

- nas **Casas do tipo Padrão** e nas **Casas de vila**, o valor será substituído por **R$ 0,00**, pois esses imóveis normalmente não apresentam cobrança de condomínio;
- nos **Apartamentos** e nas **Casas de condomínio**, o valor será substituído pela mediana do grupo formado pelas variáveis **Categoria** e **Tipo**, pois esses imóveis normalmente possuem cobrança de condomínio.

Antes de realizar a substituição definitiva, serão calculados e apresentados os valores sugeridos para cada grupo. Essa verificação permitirá avaliar a coerência das medianas e garantir que nenhuma alteração seja realizada sem análise prévia.

In [ ]:
# Selecionar registros com Condomínio igual a R$ 1,00
condicao_condominio_um = df_tratado['Condomínio'] == 1

# Calcular medianas por Categoria e Tipo,
# desconsiderando os valores simbólicos iguais a R$ 1,00
medianas_sem_valor_um = (
    df_tratado[df_tratado['Condomínio'] != 1]
    .groupby(['Categoria', 'Tipo'])['Condomínio']
    .median()
)

# Criar tabela de análise dos grupos
analise_substituicao_condominio_um = (
    df_tratado.loc[
        condicao_condominio_um,
        ['Categoria', 'Tipo']
    ]
    .value_counts()
    .rename('Quantidade')
    .reset_index()
)

# Associar a mediana de cada grupo
analise_substituicao_condominio_um['Mediana sugerida'] = (
    analise_substituicao_condominio_um.apply(
        lambda linha: medianas_sem_valor_um.get(
            (linha['Categoria'], linha['Tipo']),
            np.nan
        ),
        axis=1
    )
)

# Aplicar zero como sugestão para casas sem indicação de condomínio
condicao_casas_sem_condominio = (
    (analise_substituicao_condominio_um['Categoria'] == 'Casas') &
    (
        analise_substituicao_condominio_um['Tipo']
        .isin(['Padrão', 'Casa de vila'])
    )
)

analise_substituicao_condominio_um.loc[
    condicao_casas_sem_condominio,
    'Mediana sugerida'
] = 0

# Exibir os valores sugeridos, sem alterar df_tratado
analise_substituicao_condominio_um


,Categoria,Tipo,Quantidade,Mediana sugerida
0,Casas,Padrão,308,0.00
1,Apartamentos,Padrão,211,983.00
2,Casas,Casa de condomínio,15,200.00
3,Apartamentos,Cobertura,4,"2,300.00"
4,Apartamentos,Kitnet,3,401.00
5,Apartamentos,Loft,3,758.00
6,Casas,Casa de vila,1,0.00


#### Aplicação do tratamento dos valores iguais a R$ 1,00

Os valores sugeridos foram considerados coerentes com as características de cada grupo. Não foram identificados grupos sem uma mediana disponível.

Dessa forma, os **545 registros** com Condomínio igual a **R$ 1,00** serão tratados da seguinte maneira:

- Casas do tipo Padrão e Casas de vila receberão o valor **R$ 0,00**;
- Apartamentos e Casas de condomínio receberão a mediana de Condomínio do respectivo grupo de Categoria e Tipo;
- os valores iguais a R$ 1,00 serão desconsiderados no cálculo das medianas, evitando que valores simbólicos influenciem o preenchimento.

Após a substituição, será verificado se ainda existem registros com Condomínio igual a R$ 1,00.


In [ ]:
# Calcular as medianas por Categoria e Tipo,
# desconsiderando os valores simbólicos iguais a R$ 1,00
medianas_sem_valor_um = (
    df_tratado[df_tratado['Condomínio'] != 1]
    .groupby(['Categoria', 'Tipo'])['Condomínio']
    .median()
)

# Identificar Casas Padrão e Casas de vila com valor simbólico
condicao_casas_sem_condominio = (
    (df_tratado['Condomínio'] == 1) &
    (df_tratado['Categoria'] == 'Casas') &
    (df_tratado['Tipo'].isin(['Padrão', 'Casa de vila']))
)

# Substituir o valor simbólico por zero
df_tratado.loc[
    condicao_casas_sem_condominio,
    'Condomínio'
] = 0

# Identificar os demais registros com Condomínio igual a R$ 1,00
condicao_demais_imoveis = df_tratado['Condomínio'] == 1

# Substituir pela mediana do respectivo grupo
df_tratado.loc[
    condicao_demais_imoveis,
    'Condomínio'
] = df_tratado.loc[
    condicao_demais_imoveis
].apply(
    lambda linha: medianas_sem_valor_um.loc[
        (linha['Categoria'], linha['Tipo'])
    ],
    axis=1
)

# Validar o tratamento
print(
    f'Registros tratados: '
    f'{condicao_casas_sem_condominio.sum() + condicao_demais_imoveis.sum()}'
)

print(
    f'Valores iguais a R$ 1,00 após o tratamento: '
    f'{(df_tratado["Condomínio"] == 1).sum()}'
)

print(
    f'Registros com Condomínio igual a zero: '
    f'{(df_tratado["Condomínio"] == 0).sum()}'
)

Registros tratados: 545
Valores iguais a R$ 1,00 após o tratamento: 0
Registros com Condomínio igual a zero: 5432


#### Investigação dos valores extremamente elevados de Condomínio

Após o tratamento dos valores simbólicos iguais a **R$ 1,00**, será realizada uma análise específica dos valores extremamente elevados da variável **Condomínio**.

A análise descritiva anterior mostrou que **99,9% dos registros apresentam valores de até aproximadamente R$ 28.222,89**, enquanto alguns registros possuem valores muito superiores, chegando a milhões de reais. Esse salto indica possíveis erros de preenchimento, problemas de escala ou valores inseridos incorretamente.

Entretanto, valores elevados não serão considerados inválidos automaticamente, pois imóveis de alto padrão podem apresentar taxas condominiais superiores às encontradas na maior parte da base.

Para apoiar a decisão, serão analisados:

- a quantidade de registros acima do percentil 99,9%;
- a relação percentual entre Condomínio e preço do imóvel;
- as características dos imóveis associados a esses valores;
- possíveis erros de escala, como valores multiplicados por 100 ou 1.000;
- a existência de valores de Condomínio iguais ou superiores ao preço anunciado do imóvel.

Inicialmente, será realizada somente a investigação desses registros. Nenhum valor será alterado antes da definição de critérios objetivos de tratamento.

In [ ]:
# Calcular o limite correspondente ao percentil 99,9%
limite_condominio = df_tratado['Condomínio'].quantile(0.999)

# Selecionar registros acima do percentil 99,9%
condominios_extremos = df_tratado[
    df_tratado['Condomínio'] > limite_condominio
].copy()

# Calcular a relação entre Condomínio e preço do imóvel
condominios_extremos['Condomínio / preço (%)'] = (
    condominios_extremos['Condomínio'] /
    condominios_extremos['price'] * 100
).round(2)

print(
    f'Limite do percentil 99,9%: '
    f'R$ {limite_condominio:,.2f}'
)

print(
    f'Registros acima do percentil 99,9%: '
    f'{len(condominios_extremos)}'
)

print(
    f'Registros com Condomínio igual ou superior ao preço: '
    f'{(condominios_extremos["Condomínio"] >= condominios_extremos["price"]).sum()}'
)

# Exibir os registros em ordem decrescente
condominios_extremos[
    [
        'Condomínio',
        'price',
        'Condomínio / preço (%)',
        'Categoria',
        'Tipo',
        'Área útil',
        'Quartos',
        'Banheiros',
        'bairro',
        'ZONA'
    ]
].sort_values(
    by='Condomínio',
    ascending=False
)

Limite do percentil 99,9%: R$ 28,222.89
Registros acima do percentil 99,9%: 22
Registros com Condomínio igual ou superior ao preço: 10


,Condomínio,price,Condomínio / preço (%),Categoria,Tipo,Área útil,Quartos,Banheiros,bairro,ZONA
12233,"11,800,000.00",18000000,65.56,Apartamentos,Cobertura,1181,5,5,Santa Cecília,CENTRO
16781,"2,800,000.00",1299710,215.43,Apartamentos,Cobertura,242,3,5,Água Fria,NORTE
9921,"1,960,000.00",730000,268.49,Apartamentos,Padrão,163,3,2,Jardim São Paulo,NORTE
8937,"1,775,338.00",999240,177.67,Apartamentos,Padrão,70,2,2,Santa Cecília,CENTRO
5727,"1,200,000.00",2357000,50.91,Apartamentos,Padrão,151,3,3,Água Branca,OESTE
6314,"1,195,000.00",1195000,100.00,Apartamentos,Padrão,84,2,1,Vila Nova Conceição,SUL
5720,"1,000,000.00",2338000,42.77,Apartamentos,Padrão,151,3,3,Água Branca,OESTE
11676,"930,000.00",860000,108.14,Apartamentos,Padrão,72,3,2,Santa Teresinha,NORTE
20401,"850,000.00",500000,170.00,Apartamentos,Padrão,67,2,2,Santana,NORTE
15331,"825,000.00",400000,206.25,Apartamentos,Padrão,74,2,2,Bela Vista,CENTRO


#### Comparação dos valores extremos com imóveis semelhantes

A investigação identificou **22 registros** com valores de Condomínio acima do percentil **99,9%**, cujo limite corresponde a aproximadamente **R$ 28.222,89**. Entre esses registros, **10 apresentam valor de Condomínio igual ou superior ao preço anunciado do imóvel**, indicando forte possibilidade de erro.

Também foram observados valores de Condomínio que representam uma parcela muito elevada do preço do imóvel. Essas ocorrências podem estar relacionadas a erros de escala, valores anuais registrados como mensais, informações inseridas no campo incorreto ou outros problemas de preenchimento.

Como não foi identificado um padrão único de erro, não será aplicada uma divisão automática por 100 ou 1.000. Essa abordagem poderia corrigir alguns registros, mas produzir valores inadequados em outros.

Na próxima etapa, cada registro extremo será comparado com imóveis semelhantes, considerando:

- **Categoria**;
- **Tipo**;
- **bairro**;
- uma faixa de **Área útil**.

Além da mediana de Condomínio dos imóveis semelhantes, será apresentada a quantidade de registros utilizada em cada comparação. Essa quantidade permitirá avaliar se a estimativa possui suporte suficiente para ser utilizada no tratamento.

Os 22 valores extremos serão desconsiderados no cálculo das medianas, evitando que os próprios valores suspeitos influenciem as estimativas. Nenhuma alteração será realizada antes da avaliação dos resultados.

In [ ]:
# Definir as faixas de Área útil
faixas_area = [
    0, 30, 50, 75, 100, 150, 200, 300, 500,
    1000, float('inf')
]

rotulos_area = [
    'Até 30 m²',
    '31 a 50 m²',
    '51 a 75 m²',
    '76 a 100 m²',
    '101 a 150 m²',
    '151 a 200 m²',
    '201 a 300 m²',
    '301 a 500 m²',
    '501 a 1000 m²',
    'Acima de 1000 m²'
]

# Criar uma cópia auxiliar da base
base_com_faixa = df_tratado.copy()

# Classificar os imóveis conforme a faixa de Área útil
base_com_faixa['Faixa de área'] = pd.cut(
    base_com_faixa['Área útil'],
    bins=faixas_area,
    labels=rotulos_area,
    include_lowest=True
)

# Calcular o limite correspondente ao percentil 99,9%
limite_condominio = base_com_faixa[
    'Condomínio'
].quantile(0.999)

# Criar uma base de referência sem os valores extremos
base_referencia_condominio = base_com_faixa[
    base_com_faixa['Condomínio'] <= limite_condominio
].copy()

# Calcular a quantidade e a mediana por grupo
estatisticas_condominio_grupos = (
    base_referencia_condominio
    .groupby(
        [
            'Categoria',
            'Tipo',
            'bairro',
            'Faixa de área'
        ],
        observed=True
    )['Condomínio']
    .agg(
        Quantidade_grupo='count',
        Mediana_sugerida='median'
    )
)

# Selecionar os registros acima do percentil 99,9%
analise_condominios_extremos = base_com_faixa[
    base_com_faixa['Condomínio'] > limite_condominio
].copy()

# Associar as estatísticas dos grupos aos registros extremos
analise_condominios_extremos = (
    analise_condominios_extremos
    .join(
        estatisticas_condominio_grupos,
        on=[
            'Categoria',
            'Tipo',
            'bairro',
            'Faixa de área'
        ]
    )
)

# Calcular a relação percentual entre Condomínio e preço
analise_condominios_extremos[
    'Condomínio / preço (%)'
] = (
    analise_condominios_extremos['Condomínio'] /
    analise_condominios_extremos['price'] *
    100
).round(2)

# Organizar as colunas para apresentação
resultado_condominios_extremos = (
    analise_condominios_extremos[
        [
            'Condomínio',
            'price',
            'Condomínio / preço (%)',
            'Categoria',
            'Tipo',
            'Área útil',
            'Faixa de área',
            'bairro',
            'Quantidade_grupo',
            'Mediana_sugerida'
        ]
    ]
    .sort_values(
        by='Condomínio',
        ascending=False
    )
)

# Validar e exibir o resultado
print(
    f'Limite do percentil 99,9%: '
    f'R$ {limite_condominio:,.2f}'
)

print(
    f'Registros incluídos na comparação: '
    f'{len(resultado_condominios_extremos)}'
)

print(
    f'Registros sem grupo semelhante disponível: '
    f'{resultado_condominios_extremos["Mediana_sugerida"].isna().sum()}'
)

display(resultado_condominios_extremos)

Limite do percentil 99,9%: R$ 28,222.89
Registros incluídos na comparação: 22
Registros sem grupo semelhante disponível: 3


,Condomínio,price,Condomínio / preço (%),Categoria,Tipo,Área útil,Faixa de área,bairro,Quantidade_grupo,Mediana_sugerida
12233,"11,800,000.00",18000000,65.56,Apartamentos,Cobertura,1181,Acima de 1000 m²,Santa Cecília,1.00,0.00
16781,"2,800,000.00",1299710,215.43,Apartamentos,Cobertura,242,201 a 300 m²,Água Fria,NaN,NaN
9921,"1,960,000.00",730000,268.49,Apartamentos,Padrão,163,151 a 200 m²,Jardim São Paulo,1.00,"1,600.00"
8937,"1,775,338.00",999240,177.67,Apartamentos,Padrão,70,51 a 75 m²,Santa Cecília,31.00,652.00
5727,"1,200,000.00",2357000,50.91,Apartamentos,Padrão,151,151 a 200 m²,Água Branca,30.00,"1,541.50"
6314,"1,195,000.00",1195000,100.00,Apartamentos,Padrão,84,76 a 100 m²,Vila Nova Conceição,22.00,"1,430.50"
5720,"1,000,000.00",2338000,42.77,Apartamentos,Padrão,151,151 a 200 m²,Água Branca,30.00,"1,541.50"
11676,"930,000.00",860000,108.14,Apartamentos,Padrão,72,51 a 75 m²,Santa Teresinha,27.00,840.00
20401,"850,000.00",500000,170.00,Apartamentos,Padrão,67,51 a 75 m²,Santana,104.00,740.00
15331,"825,000.00",400000,206.25,Apartamentos,Padrão,74,51 a 75 m²,Bela Vista,143.00,750.00


#### Tratamento dos valores extremos de Condomínio

A análise identificou **22 registros** com valores de Condomínio acima do percentil **99,9%**, correspondente a aproximadamente **R$ 28.222,89**. Esses valores apresentam forte indício de erro de preenchimento ou de escala.

Entre os registros analisados, **19 possuem um grupo de comparação disponível**, formado pelas variáveis **Categoria**, **Tipo**, **bairro** e **faixa de Área útil**. Nesses casos, o valor extremo será substituído pela mediana de Condomínio do respectivo grupo.

Os **3 registros restantes** não possuem imóveis semelhantes suficientes no agrupamento utilizado. Como não existe uma base de comparação adequada para estimar o Condomínio, esses registros serão removidos da base tratada.

Após o tratamento, serão verificados:

- a quantidade de registros corrigidos;
- a quantidade de registros removidos;
- o novo total de registros da base;
- a existência de valores de Condomínio acima do limite estabelecido;
- as novas estatísticas descritivas da variável.

Essa estratégia permite preservar os imóveis que possuem uma referência disponível e remover somente os registros cuja correção não pode ser realizada com base nos dados existentes.

In [ ]:
# Guardar o total de registros antes do tratamento
registros_antes = len(df_tratado)

# Identificar os registros extremos com mediana disponível
indices_corrigir = analise_condominios_extremos[
    analise_condominios_extremos['Mediana_sugerida'].notna()
].index

# Identificar os registros extremos sem mediana disponível
indices_remover = analise_condominios_extremos[
    analise_condominios_extremos['Mediana_sugerida'].isna()
].index

# Registrar as quantidades antes das alterações
quantidade_corrigir = len(indices_corrigir)
quantidade_remover = len(indices_remover)

# Substituir os valores extremos pelas medianas sugeridas
df_tratado.loc[
    indices_corrigir,
    'Condomínio'
] = analise_condominios_extremos.loc[
    indices_corrigir,
    'Mediana_sugerida'
]

# Remover os registros sem grupo de comparação
df_tratado = df_tratado.drop(
    index=indices_remover
).copy()

# Reorganizar o índice da base
df_tratado.reset_index(
    drop=True,
    inplace=True
)

# Verificar valores que permanecem acima do limite original
valores_acima_limite = (
    df_tratado['Condomínio'] > limite_condominio
).sum()

# Exibir os resultados do tratamento
print(f'Registros corrigidos: {quantidade_corrigir}')
print(f'Registros removidos: {quantidade_remover}')
print(f'Registros antes do tratamento: {registros_antes}')
print(f'Registros após o tratamento: {len(df_tratado)}')

print(
    f'Valores de Condomínio acima do limite original: '
    f'{valores_acima_limite}'
)

print('\nEstatísticas de Condomínio após o tratamento:')
print(df_tratado['Condomínio'].describe())

Registros corrigidos: 19
Registros removidos: 3
Registros antes do tratamento: 21648
Registros após o tratamento: 21645
Valores de Condomínio acima do limite original: 0

Estatísticas de Condomínio após o tratamento:
count   21,645.00
mean     1,068.03
std      1,312.68
min          0.00
25%          0.00
50%        746.00
75%      1,460.00
max     25,000.00
Name: Condomínio, dtype: float64


#### Conclusão do tratamento da variável Condomínio

O tratamento da variável **Condomínio** foi concluído após a correção dos valores ausentes, simbólicos e extremos.

Inicialmente, os valores ausentes foram preenchidos com base na mediana dos grupos formados por **Categoria** e **Tipo**, enquanto casos específicos de casas sem cobrança condominial receberam o valor zero.

Em seguida, os **545 registros** com Condomínio igual a **R$ 1,00** foram considerados preenchimentos simbólicos. Esses valores foram substituídos por zero nas Casas Padrão e Casas de vila, enquanto os demais imóveis receberam a mediana de seus respectivos grupos.

A análise dos valores extremos identificou **22 registros** acima do percentil 99,9%. Entre eles, **19 foram corrigidos** pela mediana de imóveis semelhantes e **3 foram removidos** por não possuírem uma base de comparação disponível.

Após o tratamento:

- a base passou a apresentar **21.645 registros**;
- nenhum valor permaneceu acima do limite original de aproximadamente **R$ 28.222,89**;
- o valor máximo de Condomínio passou a ser **R$ 25.000,00**;
- a média foi reduzida para aproximadamente **R$ 1.068,03**;
- o desvio-padrão foi reduzido para aproximadamente **R$ 1.312,68**.

Esses resultados indicam que os valores inconsistentes exerciam forte influência sobre a distribuição da variável. Após as correções, a variável **Condomínio** apresenta uma distribuição mais coerente e adequada para as próximas etapas do tratamento dos dados.

### 9.2. Investigação da variável price

Após a conclusão do tratamento da variável **Condomínio**, será iniciada a análise da variável **price**, que representa o preço anunciado dos imóveis e será utilizada como variável-alvo na etapa de modelagem.

Por exercer influência direta sobre o treinamento e a avaliação dos modelos, essa variável exige uma análise cuidadosa. Valores muito baixos ou muito elevados não serão considerados erros automaticamente, pois podem representar diferentes tipos de imóveis e segmentos do mercado imobiliário.

Nesta etapa, serão investigados:

- valores iguais ou próximos de zero;
- preços excessivamente baixos;
- preços extremamente elevados;
- possíveis erros de preenchimento ou escala;
- distribuição dos preços e seus principais percentis;
- compatibilidade entre preço, Categoria, Tipo, Área útil e localização;
- relação entre preço e Área útil, por meio do valor por metro quadrado.

Inicialmente, será realizada apenas uma análise descritiva da variável. Nenhum registro será corrigido ou removido antes da definição de critérios objetivos de tratamento.

In [ ]:
# Estatísticas descritivas da variável price
print('Estatísticas descritivas de price:')
print(df_tratado['price'].describe())

# Verificar valores baixos ou inválidos
print('\nValores iguais ou inferiores a zero:')
print((df_tratado['price'] <= 0).sum())

print('\nValores inferiores a R$ 10.000,00:')
print((df_tratado['price'] < 10_000).sum())

print('\nValores inferiores a R$ 50.000,00:')
print((df_tratado['price'] < 50_000).sum())

# Verificar percentis relevantes
print('\nPercentis da variável price:')
print(
    df_tratado['price'].quantile(
        [
            0.001,
            0.005,
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
            0.995,
            0.999,
            1.00
        ]
    )
)

# Exibir os 20 menores preços
print('\n20 menores valores de price:')

display(
    df_tratado.nsmallest(
        20,
        'price'
    )[
        [
            'price',
            'Categoria',
            'Tipo',
            'Área útil',
            'Quartos',
            'Banheiros',
            'Vagas na garagem',
            'bairro',
            'ZONA'
        ]
    ]
)

# Exibir os 20 maiores preços
print('\n20 maiores valores de price:')

display(
    df_tratado.nlargest(
        20,
        'price'
    )[
        [
            'price',
            'Categoria',
            'Tipo',
            'Área útil',
            'Quartos',
            'Banheiros',
            'Vagas na garagem',
            'bairro',
            'ZONA'
        ]
    ]
)

Estatísticas descritivas de price:
count       21,645.00
mean     1,353,546.48
std      1,736,410.92
min              0.00
25%        500,000.00
50%        900,000.00
75%      1,600,000.00
max     45,900,000.00
Name: price, dtype: float64

Valores iguais ou inferiores a zero:
1

Valores inferiores a R$ 10.000,00:
20

Valores inferiores a R$ 50.000,00:
29

Percentis da variável price:
0.00       22,084.00
0.01      128,766.00
0.01      170,000.00
0.05      259,014.00
0.25      500,000.00
0.50      900,000.00
0.75    1,600,000.00
0.95    3,720,000.00
0.99    7,582,740.12
0.99   11,000,015.00
1.00   21,183,874.00
1.00   45,900,000.00
Name: price, dtype: float64

20 menores valores de price:


,price,Categoria,Tipo,Área útil,Quartos,Banheiros,Vagas na garagem,bairro,ZONA
776,0,Casas,Padrão,700,5,5,0,Vila Madalena,OESTE
13380,1,Apartamentos,Padrão,50,2,1,2,Vila Conceição,LESTE
12648,30,Casas,Casa de vila,9,2,1,5,Ponte Pequena,CENTRO
9608,35,Casas,Padrão,30,1,1,0,Jardim Boa Vista,OESTE
3448,190,Casas,Casa de vila,36,2,2,0,Engenheiro Goulart,LESTE
885,235,Apartamentos,Padrão,0,2,1,1,Jardim Mitsutani,SUL
664,250,Casas,Casa de vila,125,4,3,2,Jardim Varginha,SUL
10818,270,Casas,Casa de vila,180,3,3,1,Jardim Russo,NORTE
17607,280,Casas,Padrão,100,3,2,2,Vila Zilda,NORTE
15840,282,Apartamentos,Padrão,48,2,1,1,Vila Campanela,LESTE



20 maiores valores de price:


,price,Categoria,Tipo,Área útil,Quartos,Banheiros,Vagas na garagem,bairro,ZONA
1856,45900000,Apartamentos,Cobertura,980,3,5,5,Paraíso,SUL
5602,44784000,Apartamentos,Cobertura,720,4,5,5,Moema,SUL
3757,40000000,Casas,Padrão,1514,5,5,5,Jardim Europa,SUL
10284,34000000,Apartamentos,Padrão,753,4,5,0,Jardim Panorama,LESTE
15021,34000000,Apartamentos,Padrão,736,4,5,5,Jardim Paulistano,SUL
396,32000000,Casas,Padrão,850,3,5,5,Jardim América,SUL
8614,31000000,Casas,Padrão,1117,4,5,5,Jardim América,SUL
11955,28000000,Apartamentos,Padrão,750,5,5,5,Cidade Jardim,SUL
6988,27499990,Apartamentos,Padrão,410,3,5,5,Jardim Europa,SUL
9491,27499980,Apartamentos,Padrão,410,3,5,5,Jardim Europa,SUL


#### Investigação dos preços inferiores a R$ 50.000,00

A análise descritiva da variável **price** identificou **29 registros** com preços inferiores a **R$ 50.000,00**, incluindo um registro com preço igual a zero.

Os primeiros resultados indicam que parte desses valores pode não representar preços válidos de venda. Alguns registros apresentam valores compatíveis com aluguel mensal, enquanto outros podem representar valores simbólicos, erros de escala ou informações inseridas incorretamente.

Antes de definir qualquer tratamento, os 29 registros serão examinados individualmente, considerando:

- o preço anunciado;
- a Categoria e o Tipo do imóvel;
- a Área útil;
- a quantidade de Quartos, Banheiros e Vagas na garagem;
- o valor do Condomínio;
- o bairro e a Zona;
- outras características disponíveis que possam auxiliar na interpretação do anúncio.

Inicialmente, será realizada somente a investigação desses registros. Nenhum preço será corrigido

In [ ]:
# Selecionar apenas os registros entre R$ 10.000,00 e R$ 50.000,00
precos_10mil_50mil = df_tratado[
    (df_tratado['price'] >= 10_000) &
    (df_tratado['price'] < 50_000)
].copy()

print(
    f'Registros com preços entre R$ 10.000,00 e R$ 50.000,00: '
    f'{len(precos_10mil_50mil)}'
)

# Exibir as características relevantes
display(
    precos_10mil_50mil[
        [
            'price',
            'Categoria',
            'Tipo',
            'Área útil',
            'Quartos',
            'Banheiros',
            'Vagas na garagem',
            'Condomínio',
            'bairro',
            'ZONA'
        ]
    ].sort_values(
        by='price',
        ascending=True
    )
)

Registros com preços entre R$ 10.000,00 e R$ 50.000,00: 9


,price,Categoria,Tipo,Área útil,Quartos,Banheiros,Vagas na garagem,Condomínio,bairro,ZONA
12533,13500,Casas,Padrão,50,2,1,1,0.00,Conjunto Residencial Sitio Oratório,LESTE
892,15000,Casas,Padrão,140,2,2,0,0.00,Vila das Belezas,SUL
3918,26000,Apartamentos,Padrão,45,2,1,0,976.50,Cidade Tiradentes,LESTE
6336,30000,Casas,Padrão,100,2,1,1,0.00,Vila Pirituba,NORTE
11384,33116,Apartamentos,Padrão,42,1,1,0,0.00,Conjunto Habitacional Castro Alves,LESTE
250,40000,Apartamentos,Duplex ou triplex,123,3,2,0,"1,325.00",Bom Retiro,CENTRO
9787,40000,Casas,Padrão,46,2,1,0,0.00,Jardim Mutinga,OESTE
15635,45000,Casas,Casa de vila,48,2,2,0,0.00,Conjunto Promorar Rio Claro,LESTE
2969,48000,Apartamentos,Padrão,46,2,1,1,140.00,Cidade Tiradentes,LESTE


#### Decisão sobre os preços inferiores a R$ 50.000,00

A investigação da variável **price** identificou **29 registros** com preços inferiores a **R$ 50.000,00**, dos quais **20 apresentam valores abaixo de R$ 10.000,00**.

A análise das características desses imóveis mostrou que os valores registrados são incompatíveis com preços de venda. Foram encontrados imóveis com áreas, quantidade de quartos e demais características que indicam valores de mercado consideravelmente superiores aos preços informados.

Esses registros podem representar:

- anúncios de aluguel inseridos em uma base de imóveis para venda;
- valores de entrada ou parcelas de financiamento;
- valores simbólicos;
- erros de escala ou de preenchimento;
- informações incompletas ou registradas no campo incorreto.

Como **price** será utilizada como variável-alvo na etapa de modelagem, não será realizado o preenchimento desses registros por média ou mediana. A imputação criaria preços artificiais, reduziria a variabilidade natural dos dados e poderia introduzir distorções no treinamento e na avaliação dos modelos.

Também não foi identificado um padrão único que permita corrigir esses valores com segurança. Por esse motivo, os **29 registros com preço inferior a R$ 50.000,00 serão removidos** da base tratada.

Esses registros representam aproximadamente **0,13% da base atual**, portanto sua remoção terá impacto quantitativo reduzido e contribuirá para preservar a confiabilidade da variável-alvo.



In [ ]:
# Identificar os registros com preço inferior a R$ 50.000,00
condicao_preco_invalido = df_tratado['price'] < 50_000

# Registrar as quantidades antes da remoção
registros_antes = len(df_tratado)
registros_removidos = condicao_preco_invalido.sum()

# Remover os registros com preços incompatíveis
df_tratado = df_tratado[
    ~condicao_preco_invalido
].copy()

# Reorganizar o índice
df_tratado.reset_index(
    drop=True,
    inplace=True
)

# Validar a remoção
print(f'Registros removidos: {registros_removidos}')
print(f'Registros antes da remoção: {registros_antes}')
print(f'Registros após a remoção: {len(df_tratado)}')
print(f'Menor preço após o tratamento: R$ {df_tratado["price"].min():,.2f}')
print(
    f'Preços abaixo de R$ 50.000,00 após o tratamento: '
    f'{(df_tratado["price"] < 50_000).sum()}'
)

Registros removidos: 29
Registros antes da remoção: 21645
Registros após a remoção: 21616
Menor preço após o tratamento: R$ 50,000.00
Preços abaixo de R$ 50.000,00 após o tratamento: 0


#### Investigação dos preços elevados

Após a remoção dos registros com preços inferiores a **R$ 50.000,00**, será concluída a análise da variável **price** por meio da investigação dos valores mais elevados.

Diferentemente dos preços muito baixos, os valores elevados não serão considerados inconsistentes automaticamente. Imóveis de alto padrão, localizados em regiões valorizadas e com grandes áreas, podem apresentar preços significativamente superiores aos observados na maior parte da base.

Para avaliar esses registros, serão considerados:

- os preços acima do percentil **99,9%**;
- a Categoria e o Tipo do imóvel;
- a Área útil;
- a localização;
- a quantidade de Quartos, Banheiros e Vagas na garagem;
- o preço por metro quadrado.

O preço por metro quadrado permitirá comparar imóveis de diferentes tamanhos e identificar casos em que o preço elevado é compatível com a área e a localização. Registros com Área útil igual a zero não terão essa relação calculada, pois essa divisão produziria um resultado inválido.

Inicialmente, os imóveis com preços mais elevados serão apenas identificados e comparados. Nenhum registro será removido ou corrigido antes da avaliação dos resultados.

In [ ]:
# Calcular o limite correspondente ao percentil 99,9% de price
limite_preco = df_tratado['price'].quantile(0.999)

# Selecionar os registros acima do percentil 99,9%
precos_elevados = df_tratado[
    df_tratado['price'] > limite_preco
].copy()

# Calcular o preço por metro quadrado
# Relação utilizada: price dividido pela Área útil
precos_elevados['Preço por m²'] = np.where(
    precos_elevados['Área útil'] > 0,
    precos_elevados['price'] / precos_elevados['Área útil'],
    np.nan
)

# Arredondar o preço por metro quadrado
precos_elevados['Preço por m²'] = (
    precos_elevados['Preço por m²'].round(2)
)

# Função para formatar valores no padrão brasileiro
def formatar_valor_brasileiro(valor):
    if pd.isna(valor):
        return ''

    return (
        f'{valor:,.2f}'
        .replace(',', 'X')
        .replace('.', ',')
        .replace('X', '.')
    )

# Calcular indicadores do preço por metro quadrado
preco_m2_minimo = precos_elevados['Preço por m²'].min()
preco_m2_mediano = precos_elevados['Preço por m²'].median()
preco_m2_maximo = precos_elevados['Preço por m²'].max()

# Exibir informações gerais
print(
    f'Limite do percentil 99,9%: '
    f'R$ {formatar_valor_brasileiro(limite_preco)}'
)

print(
    f'Registros acima do percentil 99,9%: '
    f'{len(precos_elevados)}'
)

print(
    f'Registros com Área útil igual a zero: '
    f'{(precos_elevados["Área útil"] == 0).sum()}'
)

print(
    '\nRelação analisada: '
    'Preço por m² = price ÷ Área útil'
)

print(
    f'Menor preço por m² entre os registros analisados: '
    f'R$ {formatar_valor_brasileiro(preco_m2_minimo)}'
)

print(
    f'Mediana do preço por m² entre os registros analisados: '
    f'R$ {formatar_valor_brasileiro(preco_m2_mediano)}'
)

print(
    f'Maior preço por m² entre os registros analisados: '
    f'R$ {formatar_valor_brasileiro(preco_m2_maximo)}'
)

# Organizar os registros para apresentação
resultado_precos_elevados = (
    precos_elevados[
        [
            'price',
            'Preço por m²',
            'Categoria',
            'Tipo',
            'Área útil',
            'Quartos',
            'Banheiros',
            'Vagas na garagem',
            'Condomínio',
            'bairro',
            'ZONA'
        ]
    ]
    .sort_values(
        by='price',
        ascending=False
    )
)

# Exibir com formatação monetária brasileira
display(
    resultado_precos_elevados.style.format(
        {
            'price': formatar_valor_brasileiro,
            'Preço por m²': formatar_valor_brasileiro,
            'Condomínio': formatar_valor_brasileiro
        }
    )
)

Limite do percentil 99,9%: R$ 21.198.852,50
Registros acima do percentil 99,9%: 22
Registros com Área útil igual a zero: 0

Relação analisada: Preço por m² = price ÷ Área útil
Menor preço por m² entre os registros analisados: R$ 10.600,00
Mediana do preço por m² entre os registros analisados: R$ 49.191,56
Maior preço por m² entre os registros analisados: R$ 67.073,15


,price,Preço por m²,Categoria,Tipo,Área útil,Quartos,Banheiros,Vagas na garagem,Condomínio,bairro,ZONA
1850,"45.900.000,00","46.836,73",Apartamentos,Cobertura,980,3,5,5,"17.000,00",Paraíso,SUL
5593,"44.784.000,00","62.200,00",Apartamentos,Cobertura,720,4,5,5,"12.500,00",Moema,SUL
3749,"40.000.000,00","26.420,08",Casas,Padrão,1514,5,5,5,"0,00",Jardim Europa,SUL
10268,"34.000.000,00","45.152,72",Apartamentos,Padrão,753,4,5,0,"14.781,00",Jardim Panorama,LESTE
14999,"34.000.000,00","46.195,65",Apartamentos,Padrão,736,4,5,5,"18.408,00",Jardim Paulistano,SUL
395,"32.000.000,00","37.647,06",Casas,Padrão,850,3,5,5,"0,00",Jardim América,SUL
8602,"31.000.000,00","27.752,91",Casas,Padrão,1117,4,5,5,"0,00",Jardim América,SUL
11937,"28.000.000,00","37.333,33",Apartamentos,Padrão,750,5,5,5,"12.500,00",Cidade Jardim,SUL
6977,"27.499.990,00","67.073,15",Apartamentos,Padrão,410,3,5,5,"9.500,00",Jardim Europa,SUL
9479,"27.499.980,00","67.073,12",Apartamentos,Padrão,410,3,5,5,"9.500,00",Jardim Europa,SUL


#### Conclusão do tratamento da variável price

A investigação da variável **price** identificou e removeu **29 registros** com preços inferiores a **R$ 50.000,00**, considerados incompatíveis com valores de venda e possivelmente relacionados a anúncios de aluguel, valores simbólicos ou erros de preenchimento.

Os **22 registros** acima do percentil 99,9% também foram avaliados por meio da relação entre preço e Área útil. Os preços por metro quadrado variaram entre **R$ 10.600,00 e R$ 67.073,15**, sem evidências suficientes para considerar os preços elevados como incorretos. Por esse motivo, esses registros foram mantidos.

Após o tratamento, a base permaneceu com **21.616 registros**, tendo **R$ 50.000,00** como menor preço válido. Com isso, o tratamento da variável **price** foi concluído, e a próxima etapa será dedicada à investigação da variável **Área útil**.

### 9.3. Investigação da variável Área útil

Após a conclusão do tratamento da variável **price**, será iniciada a investigação da variável **Área útil**, que representa a metragem disponível para utilização no imóvel.

Essa variável exerce influência direta sobre o preço e será importante para a construção dos modelos de previsão. Por esse motivo, valores inconsistentes podem prejudicar a análise da relação entre tamanho e valor dos imóveis.

Nesta etapa, serão investigados:

- valores iguais ou inferiores a zero;
- áreas excessivamente pequenas;
- áreas extremamente elevadas;
- possíveis erros de preenchimento ou escala;
- compatibilidade da Área útil com a Categoria e o Tipo do imóvel;
- relação entre Área útil, preço e preço por metro quadrado.

Áreas pequenas ou elevadas não serão consideradas inválidas automaticamente. Kitnets, Lofts, apartamentos compactos, casas de alto padrão e imóveis localizados em grandes terrenos podem apresentar metragens legítimas fora do padrão predominante da base.

Inicialmente, será realizada apenas uma análise descritiva. Nenhum registro será corrigido ou removido antes da definição de critérios objetivos de tratamento.

In [ ]:
# Estatísticas descritivas da variável Área útil
print('Estatísticas descritivas de Área útil:')
print(df_tratado['Área útil'].describe())

# Verificar valores baixos ou inválidos
print('\nValores iguais ou inferiores a zero:')
print((df_tratado['Área útil'] <= 0).sum())

print('\nValores entre 1 m² e 10 m²:')
print(
    (
        (df_tratado['Área útil'] >= 1) &
        (df_tratado['Área útil'] <= 10)
    ).sum()
)

print('\nValores superiores a 1.000 m²:')
print((df_tratado['Área útil'] > 1_000).sum())

# Verificar percentis relevantes
print('\nPercentis da variável Área útil:')
print(
    df_tratado['Área útil'].quantile(
        [
            0.001,
            0.005,
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99,
            0.995,
            0.999,
            1.00
        ]
    )
)

# Exibir os registros com as 20 menores áreas
print('\n20 menores valores de Área útil:')

display(
    df_tratado.nsmallest(
        20,
        'Área útil'
    )[
        [
            'Área útil',
            'price',
            'Categoria',
            'Tipo',
            'Quartos',
            'Banheiros',
            'Vagas na garagem',
            'bairro',
            'ZONA'
        ]
    ].style.format(
        {
            'price': formatar_valor_brasileiro
        }
    )
)

# Exibir os registros com as 20 maiores áreas
print('\n20 maiores valores de Área útil:')

display(
    df_tratado.nlargest(
        20,
        'Área útil'
    )[
        [
            'Área útil',
            'price',
            'Categoria',
            'Tipo',
            'Quartos',
            'Banheiros',
            'Vagas na garagem',
            'bairro',
            'ZONA'
        ]
    ].style.format(
        {
            'price': formatar_valor_brasileiro
        }
    )
)

Estatísticas descritivas de Área útil:
count    21,616.00
mean        248.89
std       7,504.90
min           0.00
25%          65.00
50%         107.00
75%         168.00
max     608,608.00
Name: Área útil, dtype: float64

Valores iguais ou inferiores a zero:
1

Valores entre 1 m² e 10 m²:
104

Valores superiores a 1.000 m²:
38

Percentis da variável Área útil:
0.00        10.00
0.01        12.00
0.01        24.00
0.05        35.00
0.25        65.00
0.50       107.00
0.75       168.00
0.95       330.00
0.99       579.25
0.99       714.40
1.00     1,338.48
1.00   608,608.00
Name: Área útil, dtype: float64

20 menores valores de Área útil:


,Área útil,price,Categoria,Tipo,Quartos,Banheiros,Vagas na garagem,bairro,ZONA
429,0,"90.000,00",Apartamentos,Padrão,2,1,1,Jardim Pedra Branca,NORTE
8093,1,"130.000,00",Casas,Padrão,2,1,0,Parque Independência,LESTE
6589,10,"211.832,00",Apartamentos,Padrão,2,1,0,Campos Elíseos,CENTRO
8879,10,"206.500,00",Apartamentos,Padrão,2,1,1,Lauzane Paulista,NORTE
8880,10,"235.200,00",Apartamentos,Padrão,2,1,1,Nossa Senhora do Ó,NORTE
8887,10,"246.772,00",Apartamentos,Padrão,3,1,1,Piqueri,NORTE
10182,10,"368.672,00",Apartamentos,Padrão,2,1,0,Campos Elíseos,CENTRO
10186,10,"296.000,00",Apartamentos,Padrão,1,1,0,Bela Vista,CENTRO
10187,10,"304.000,00",Apartamentos,Padrão,2,1,0,Bom Retiro,CENTRO
10189,10,"352.000,00",Apartamentos,Padrão,2,1,0,Campos Elíseos,CENTRO



20 maiores valores de Área útil:


,Área útil,price,Categoria,Tipo,Quartos,Banheiros,Vagas na garagem,bairro,ZONA
8576,608608,"7.000.000,00",Casas,Padrão,4,5,5,Jardim Petrópolis,SUL
9861,608608,"6.500.000,00",Casas,Padrão,4,5,5,Jardim Petrópolis,SUL
9866,608608,"6.000.000,00",Casas,Padrão,4,5,5,Jardim Petrópolis,SUL
4081,200000,"1.100.000,00",Casas,Casa de condomínio,4,3,2,Barra Funda,CENTRO
10682,200000,"1.100.000,00",Casas,Casa de condomínio,4,3,2,Barra Funda,OESTE
7368,150000,"140.000,00",Casas,Padrão,2,1,1,Jardim Limoeiro,LESTE
3942,65120,"295.000,00",Casas,Casa de condomínio,2,1,1,Vila Rui Barbosa,LESTE
15618,11020,"300.000,00",Casas,Padrão,2,2,0,Parque Císper,LESTE
9182,3336,"255.000,00",Apartamentos,Padrão,2,1,0,Vila Guedes,OESTE
2820,2662,"5.800.000,00",Apartamentos,Padrão,4,5,5,Imirim,NORTE


#### Investigação das áreas iguais ou inferiores a 10 m²

A análise descritiva identificou **105 registros** com Área útil entre **0 e 10 m²**, incluindo um imóvel com área igual a zero.

Embora imóveis compactos possam apresentar metragens reduzidas, alguns desses registros possuem dois ou três quartos, indicando possível erro de preenchimento, valor simbólico ou problema de escala.

Antes de definir o tratamento, serão analisadas a frequência de cada metragem e as características dos imóveis associados. Nenhum registro será alterado nesta etapa.

In [ ]:
# Selecionar registros com Área útil entre 0 e 10 m²
areas_ate_10 = df_tratado[
    (df_tratado['Área útil'] >= 0) &
    (df_tratado['Área útil'] <= 10)
].copy()

print(
    f'Registros com Área útil entre 0 e 10 m²: '
    f'{len(areas_ate_10)}'
)

print('\nFrequência de cada valor de Área útil:')
print(
    areas_ate_10['Área útil']
    .value_counts()
    .sort_index()
)

print('\nDistribuição por Categoria e Tipo:')
print(
    areas_ate_10.groupby(
        ['Categoria', 'Tipo']
    ).size().sort_values(ascending=False)
)

# Exibir os registros para investigação
display(
    areas_ate_10[
        [
            'Área útil',
            'price',
            'Categoria',
            'Tipo',
            'Quartos',
            'Banheiros',
            'Vagas na garagem',
            'bairro',
            'ZONA'
        ]
    ]
    .sort_values(
        by=['Área útil', 'price']
    )
    .style.format(
        {'price': formatar_valor_brasileiro}
    )
)

Registros com Área útil entre 0 e 10 m²: 105

Frequência de cada valor de Área útil:
Área útil
0       1
1       1
10    103
Name: count, dtype: int64

Distribuição por Categoria e Tipo:
Categoria     Tipo  
Apartamentos  Padrão    103
Casas         Padrão      2
dtype: int64


,Área útil,price,Categoria,Tipo,Quartos,Banheiros,Vagas na garagem,bairro,ZONA
429,0,"90.000,00",Apartamentos,Padrão,2,1,1,Jardim Pedra Branca,NORTE
8093,1,"130.000,00",Casas,Padrão,2,1,0,Parque Independência,LESTE
12282,10,"62.903,00",Apartamentos,Padrão,2,1,0,Cidade Tiradentes,LESTE
12311,10,"79.200,00",Apartamentos,Padrão,1,1,0,Jardim Santa Terezinha,LESTE
17087,10,"79.200,00",Apartamentos,Padrão,1,1,0,Jardim Santa Terezinha,LESTE
20071,10,"83.304,00",Apartamentos,Padrão,2,1,0,Chácara Santa Etelvina,LESTE
12521,10,"85.347,00",Apartamentos,Padrão,2,1,1,Jardim da Laranjeira,LESTE
12297,10,"86.400,00",Apartamentos,Padrão,2,1,0,Cidade Tiradentes,LESTE
17069,10,"86.400,00",Apartamentos,Padrão,2,1,0,Cidade Tiradentes,LESTE
12302,10,"96.600,00",Apartamentos,Padrão,2,1,0,Itaim Paulista,LESTE


#### Estratégia para o tratamento das áreas iguais ou inferiores a 10 m²

A investigação identificou **105 registros** com Área útil entre **0 e 10 m²**. Desse total, **103 imóveis apresentam exatamente 10 m²**, enquanto um registro possui área igual a zero e outro possui área igual a 1 m².

A concentração de registros em exatamente 10 m², associada à presença de imóveis com dois ou três quartos, indica que esses valores provavelmente representam preenchimentos padronizados, informações ausentes ou erros de registro.

Como as demais características desses imóveis podem ser aproveitadas, os registros não serão removidos automaticamente. Inicialmente, serão calculadas áreas sugeridas com base na mediana de imóveis semelhantes, considerando:

- **Categoria**;
- **Tipo**;
- **bairro**;
- **Quartos**;
- **Banheiros**.

Os registros com Área útil igual ou inferior a 10 m² serão desconsiderados no cálculo das medianas, evitando que os valores suspeitos influenciem as estimativas.

Antes de qualquer substituição, serão apresentados o valor sugerido e a quantidade de imóveis utilizada em cada grupo. A base somente será alterada após a avaliação da confiabilidade dos resultados.

In [ ]:
# Criar uma base de referência sem as áreas suspeitas
base_referencia_area = df_tratado[
    df_tratado['Área útil'] > 10
].copy()

# Calcular quantidade e mediana da área por grupo
estatisticas_area_grupos = (
    base_referencia_area
    .groupby(
        [
            'Categoria',
            'Tipo',
            'bairro',
            'Quartos',
            'Banheiros'
        ]
    )['Área útil']
    .agg(
        Quantidade_grupo='count',
        Mediana_sugerida='median'
    )
)

# Selecionar os registros com área igual ou inferior a 10 m²
analise_areas_ate_10 = df_tratado[
    df_tratado['Área útil'] <= 10
].copy()

# Associar as estatísticas dos imóveis semelhantes
analise_areas_ate_10 = (
    analise_areas_ate_10
    .join(
        estatisticas_area_grupos,
        on=[
            'Categoria',
            'Tipo',
            'bairro',
            'Quartos',
            'Banheiros'
        ]
    )
)

# Organizar as colunas para apresentação
resultado_areas_ate_10 = analise_areas_ate_10[
    [
        'Área útil',
        'price',
        'Categoria',
        'Tipo',
        'bairro',
        'Quartos',
        'Banheiros',
        'Quantidade_grupo',
        'Mediana_sugerida'
    ]
].sort_values(
    by=['Quantidade_grupo', 'Mediana_sugerida'],
    ascending=[False, True]
)

# Exibir o resumo da comparação
print(f'Registros analisados: {len(resultado_areas_ate_10)}')

print(
    f'Registros com mediana disponível: '
    f'{resultado_areas_ate_10["Mediana_sugerida"].notna().sum()}'
)

print(
    f'Registros sem grupo de comparação: '
    f'{resultado_areas_ate_10["Mediana_sugerida"].isna().sum()}'
)

# Exibir a tabela com preço formatado
display(
    resultado_areas_ate_10.style.format(
        {
            'price': formatar_valor_brasileiro,
            'Mediana_sugerida': '{:,.2f}'
        },
        na_rep='Sem referência'
    )
)

Registros analisados: 105
Registros com mediana disponível: 92
Registros sem grupo de comparação: 13


,Área útil,price,Categoria,Tipo,bairro,Quartos,Banheiros,Quantidade_grupo,Mediana_sugerida
10186,10,"296.000,00",Apartamentos,Padrão,Bela Vista,1,1,153.000000,39.00
10199,10,"731.505,00",Apartamentos,Padrão,Bela Vista,2,1,85.000000,55.00
17835,10,"836.000,00",Apartamentos,Padrão,Barra Funda,2,1,75.000000,49.00
17102,10,"295.000,00",Apartamentos,Padrão,Mooca,2,1,51.000000,54.00
19815,10,"400.006,00",Apartamentos,Padrão,Mooca,2,1,51.000000,54.00
17094,10,"480.000,00",Apartamentos,Padrão,Tatuapé,2,1,44.000000,53.50
8879,10,"206.500,00",Apartamentos,Padrão,Lauzane Paulista,2,1,37.000000,55.00
12171,10,"206.500,00",Apartamentos,Padrão,Lauzane Paulista,2,1,37.000000,55.00
17077,10,"191.092,00",Apartamentos,Padrão,Itaquera,2,1,30.000000,40.00
6589,10,"211.832,00",Apartamentos,Padrão,Campos Elíseos,2,1,28.000000,62.00


#### Aprimoramento da estratégia de estimativa da Área útil

O primeiro agrupamento considerou as variáveis **Categoria**, **Tipo**, **bairro**, **Quartos** e **Banheiros**. Essa análise forneceu uma mediana para **92 dos 105 registros**, mas alguns grupos apresentaram poucas referências e **13 registros** permaneceram sem uma estimativa disponível.

Para aumentar a confiabilidade do tratamento, será aplicada uma estratégia hierárquica:

1. manter a mediana do agrupamento original quando houver pelo menos **5 imóveis semelhantes**;
2. para grupos com menos de 5 referências ou sem mediana disponível, utilizar um agrupamento mais amplo;
3. o segundo agrupamento considerará **Categoria**, **Tipo**, **Quartos**, **Banheiros** e uma **faixa de preço**, sem utilizar o bairro;
4. os registros com Área útil igual ou inferior a **10 m²** permanecerão fora do cálculo das medianas;
5. as novas estimativas serão apresentadas e avaliadas antes de qualquer alteração na base.

A inclusão da faixa de preço permite diferenciar imóveis com características estruturais semelhantes, mas pertencentes a padrões de valor distintos. Nenhum registro será modificado nesta etapa.

In [ ]:
# Criar a base de referência sem áreas suspeitas
base_referencia_area = df_tratado[
    df_tratado['Área útil'] > 10
].copy()

# Definir faixas de preço a partir dos quartis da base
limites_preco = [
    0,
    df_tratado['price'].quantile(0.25),
    df_tratado['price'].quantile(0.50),
    df_tratado['price'].quantile(0.75),
    df_tratado['price'].quantile(0.90),
    float('inf')
]

rotulos_preco = [
    'Até Q1',
    'Q1 a Mediana',
    'Mediana a Q3',
    'Q3 a P90',
    'Acima de P90'
]

# Criar a faixa de preço na base de referência
base_referencia_area['Faixa de preço'] = pd.cut(
    base_referencia_area['price'],
    bins=limites_preco,
    labels=rotulos_preco,
    include_lowest=True
)

# Criar a mesma faixa na base com áreas suspeitas
analise_areas_hierarquica = analise_areas_ate_10.copy()

analise_areas_hierarquica['Faixa de preço'] = pd.cut(
    analise_areas_hierarquica['price'],
    bins=limites_preco,
    labels=rotulos_preco,
    include_lowest=True
)

# Calcular a mediana no agrupamento mais amplo
estatisticas_area_amplas = (
    base_referencia_area
    .groupby(
        [
            'Categoria',
            'Tipo',
            'Quartos',
            'Banheiros',
            'Faixa de preço'
        ],
        observed=True
    )['Área útil']
    .agg(
        Quantidade_grupo_amplo='count',
        Mediana_grupo_amplo='median'
    )
)

# Associar as estatísticas do agrupamento amplo
analise_areas_hierarquica = (
    analise_areas_hierarquica
    .join(
        estatisticas_area_amplas,
        on=[
            'Categoria',
            'Tipo',
            'Quartos',
            'Banheiros',
            'Faixa de preço'
        ]
    )
)

# Usar o grupo específico quando houver pelo menos 5 referências
condicao_grupo_especifico = (
    analise_areas_hierarquica['Quantidade_grupo'] >= 5
)

analise_areas_hierarquica['Mediana final sugerida'] = np.where(
    condicao_grupo_especifico,
    analise_areas_hierarquica['Mediana_sugerida'],
    analise_areas_hierarquica['Mediana_grupo_amplo']
)

analise_areas_hierarquica['Grupo utilizado'] = np.where(
    condicao_grupo_especifico,
    'Categoria + Tipo + Bairro + Quartos + Banheiros',
    'Categoria + Tipo + Quartos + Banheiros + Faixa de preço'
)

analise_areas_hierarquica['Quantidade final do grupo'] = np.where(
    condicao_grupo_especifico,
    analise_areas_hierarquica['Quantidade_grupo'],
    analise_areas_hierarquica['Quantidade_grupo_amplo']
)

# Organizar o resultado
resultado_areas_hierarquica = (
    analise_areas_hierarquica[
        [
            'Área útil',
            'price',
            'Categoria',
            'Tipo',
            'bairro',
            'Quartos',
            'Banheiros',
            'Faixa de preço',
            'Grupo utilizado',
            'Quantidade final do grupo',
            'Mediana final sugerida'
        ]
    ]
    .sort_values(
        by='Quantidade final do grupo',
        ascending=False
    )
)

# Validar e exibir
print(f'Registros analisados: {len(resultado_areas_hierarquica)}')

print(
    f'Registros com estimativa final disponível: '
    f'{resultado_areas_hierarquica["Mediana final sugerida"].notna().sum()}'
)

print(
    f'Registros ainda sem estimativa: '
    f'{resultado_areas_hierarquica["Mediana final sugerida"].isna().sum()}'
)

display(
    resultado_areas_hierarquica.style.format(
        {
            'price': formatar_valor_brasileiro,
            'Quantidade final do grupo': '{:,.0f}',
            'Mediana final sugerida': '{:,.2f}'
        },
        na_rep='Sem referência'
    )
)

Registros analisados: 105
Registros com estimativa final disponível: 105
Registros ainda sem estimativa: 0


,Área útil,price,Categoria,Tipo,bairro,Quartos,Banheiros,Faixa de preço,Grupo utilizado,Quantidade final do grupo,Mediana final sugerida
429,0,"90.000,00",Apartamentos,Padrão,Jardim Pedra Branca,2,1,Até Q1,Categoria + Tipo + Quartos + Banheiros + Faixa de preço,"2,196",49.00
10187,10,"304.000,00",Apartamentos,Padrão,Bom Retiro,2,1,Até Q1,Categoria + Tipo + Quartos + Banheiros + Faixa de preço,"2,196",49.00
12310,10,"220.000,00",Apartamentos,Padrão,Jardim Três Marias,2,1,Até Q1,Categoria + Tipo + Quartos + Banheiros + Faixa de preço,"2,196",49.00
12315,10,"146.900,00",Apartamentos,Padrão,Parque São Lourenço,2,1,Até Q1,Categoria + Tipo + Quartos + Banheiros + Faixa de preço,"2,196",49.00
12298,10,"130.000,00",Apartamentos,Padrão,Conjunto Residencial Sitio Oratório,2,1,Até Q1,Categoria + Tipo + Quartos + Banheiros + Faixa de preço,"2,196",49.00
12295,10,"169.575,00",Apartamentos,Padrão,Jardim Jaú,2,1,Até Q1,Categoria + Tipo + Quartos + Banheiros + Faixa de preço,"2,196",49.00
12302,10,"96.600,00",Apartamentos,Padrão,Itaim Paulista,2,1,Até Q1,Categoria + Tipo + Quartos + Banheiros + Faixa de preço,"2,196",49.00
12301,10,"122.500,00",Apartamentos,Padrão,Vila Roseira II,2,1,Até Q1,Categoria + Tipo + Quartos + Banheiros + Faixa de preço,"2,196",49.00
12089,10,"107.032,00",Apartamentos,Padrão,Vila Princesa Isabel,2,1,Até Q1,Categoria + Tipo + Quartos + Banheiros + Faixa de preço,"2,196",49.00
12379,10,"212.000,00",Apartamentos,Padrão,Jardim Brasília,2,1,Até Q1,Categoria + Tipo + Quartos + Banheiros + Faixa de preço,"2,196",49.00


In [ ]:
# Classificar as estimativas conforme o tamanho do grupo utilizado
estimativas_baixo_suporte = resultado_areas_hierarquica[
    resultado_areas_hierarquica['Quantidade final do grupo'] < 5
]

print(
    f'Estimativas baseadas em pelo menos 5 imóveis: '
    f'{(resultado_areas_hierarquica["Quantidade final do grupo"] >= 5).sum()}'
)

print(
    f'Estimativas baseadas em menos de 5 imóveis: '
    f'{len(estimativas_baixo_suporte)}'
)

# Exibir somente os casos com menor suporte
display(
    estimativas_baixo_suporte.style.format(
        {
            'price': formatar_valor_brasileiro,
            'Quantidade final do grupo': '{:,.0f}',
            'Mediana final sugerida': '{:,.2f}'
        },
        na_rep='Sem referência'
    )
)

Estimativas baseadas em pelo menos 5 imóveis: 104
Estimativas baseadas em menos de 5 imóveis: 1


,Área útil,price,Categoria,Tipo,bairro,Quartos,Banheiros,Faixa de preço,Grupo utilizado,Quantidade final do grupo,Mediana final sugerida
17068,10,"818.000,00",Apartamentos,Padrão,Vila Formosa,4,1,Q1 a Mediana,Categoria + Tipo + Quartos + Banheiros + Faixa de preço,1,140.00


#### Aplicação do tratamento das áreas iguais ou inferiores a 10 m²

A estratégia hierárquica forneceu estimativas para todos os **105 registros** com Área útil igual ou inferior a **10 m²**.

Entre esses registros, **104 obtiveram estimativas baseadas em grupos com pelo menos 5 imóveis semelhantes**, apresentando suporte suficiente para a substituição dos valores inconsistentes pelas medianas sugeridas.

Apenas **1 registro** apresentou uma estimativa baseada em um único imóvel, não oferecendo suporte suficiente para uma correção confiável. Por esse motivo, esse registro será removido da base tratada.

O tratamento será realizado da seguinte maneira:

- **104 registros** terão a Área útil substituída pela mediana final sugerida;
- **1 registro** será removido por não possuir uma base de comparação adequada;
- após o tratamento, será verificada a inexistência de áreas iguais ou inferiores a 10 m²;
- a base deverá passar de **21.616 para 21.615 registros**.

Essa estratégia permite preservar a maior parte dos imóveis e remover somente o registro cuja correção não pode ser sustentada adequadamente pelos dados disponíveis.

In [ ]:
# Identificar registros com estimativas apoiadas por pelo menos 5 imóveis
condicao_corrigir_area = (
    resultado_areas_hierarquica['Quantidade final do grupo'] >= 5
)

indices_corrigir_area = resultado_areas_hierarquica[
    condicao_corrigir_area
].index

# Identificar o registro com estimativa baseada em menos de 5 imóveis
indices_remover_area = resultado_areas_hierarquica[
    ~condicao_corrigir_area
].index

# Registrar as quantidades antes do tratamento
registros_antes = len(df_tratado)
quantidade_corrigir = len(indices_corrigir_area)
quantidade_remover = len(indices_remover_area)

# Substituir as áreas inconsistentes pelas medianas sugeridas
df_tratado.loc[
    indices_corrigir_area,
    'Área útil'
] = resultado_areas_hierarquica.loc[
    indices_corrigir_area,
    'Mediana final sugerida'
].round().astype('int64')

# Remover o registro com baixo suporte
df_tratado.drop(
    index=indices_remover_area,
    inplace=True
)

# Reorganizar o índice
df_tratado.reset_index(
    drop=True,
    inplace=True
)

# Validar o tratamento
print(f'Registros corrigidos: {quantidade_corrigir}')
print(f'Registros removidos: {quantidade_remover}')
print(f'Registros antes do tratamento: {registros_antes}')
print(f'Registros após o tratamento: {len(df_tratado)}')

print(
    f'Áreas iguais ou inferiores a 10 m² após o tratamento: '
    f'{(df_tratado["Área útil"] <= 10).sum()}'
)

print(
    f'Menor Área útil após o tratamento: '
    f'{df_tratado["Área útil"].min()} m²'
)

print(
    f'Tipo da variável Área útil: '
    f'{df_tratado["Área útil"].dtype}'
)

Registros corrigidos: 104
Registros removidos: 1
Registros antes do tratamento: 21616
Registros após o tratamento: 21615
Áreas iguais ou inferiores a 10 m² após o tratamento: 0
Menor Área útil após o tratamento: 12 m²
Tipo da variável Área útil: int64


#### Resultado do tratamento das áreas iguais ou inferiores a 10 m²

O tratamento corrigiu **104 registros** por meio das medianas obtidas em grupos de imóveis semelhantes. Um registro foi removido por possuir uma estimativa baseada em apenas um imóvel, não apresentando suporte suficiente para uma correção confiável.

Após o tratamento, a base passou a apresentar **21.615 registros**, sem valores de Área útil iguais ou inferiores a 10 m². O menor valor registrado passou a ser **12 m²**, e a variável permaneceu no formato inteiro (`int64`).

#### Investigação das áreas extremamente elevadas

Na sequência, serão investigados os registros com valores extremamente elevados de **Área útil**.

A análise anterior identificou valores como **608.608 m², 200.000 m², 150.000 m² e 65.120 m²**, que são incompatíveis com as características dos imóveis apresentados e podem representar erros de preenchimento, escala ou separador decimal.

Entretanto, áreas superiores a 1.000 m² não serão consideradas inválidas automaticamente, pois imóveis de alto padrão, grandes residências e propriedades localizadas em terrenos extensos podem apresentar metragens legítimas.

A investigação considerará:

- o limite correspondente ao percentil 99,9%;
- o preço do imóvel;
- o preço por metro quadrado;
- a Categoria e o Tipo;
- a quantidade de Quartos, Banheiros e Vagas na garagem;
- o bairro e a Zona;
- a comparação com imóveis semelhantes.

Inicialmente, será realizada apenas a identificação dos registros suspeitos. Nenhuma área será corrigida ou removida antes da definição de critérios objetivos de tratamento.

In [ ]:
# Calcular o limite correspondente ao percentil 99,9%
limite_area = df_tratado['Área útil'].quantile(0.999)

# Selecionar os registros acima do percentil 99,9%
areas_elevadas = df_tratado[
    df_tratado['Área útil'] > limite_area
].copy()

# Calcular o preço por metro quadrado
areas_elevadas['Preço por m²'] = (
    areas_elevadas['price'] /
    areas_elevadas['Área útil']
).round(2)

# Exibir informações gerais
print(
    f'Limite do percentil 99,9%: '
    f'{formatar_valor_brasileiro(limite_area)} m²'
)

print(
    f'Registros acima do percentil 99,9%: '
    f'{len(areas_elevadas)}'
)

print(
    f'Menor área entre os registros investigados: '
    f'{formatar_valor_brasileiro(areas_elevadas["Área útil"].min())} m²'
)

print(
    f'Maior área entre os registros investigados: '
    f'{formatar_valor_brasileiro(areas_elevadas["Área útil"].max())} m²'
)

# Organizar as informações para apresentação
resultado_areas_elevadas = (
    areas_elevadas[
        [
            'Área útil',
            'price',
            'Preço por m²',
            'Categoria',
            'Tipo',
            'Quartos',
            'Banheiros',
            'Vagas na garagem',
            'Condomínio',
            'bairro',
            'ZONA'
        ]
    ]
    .sort_values(
        by='Área útil',
        ascending=False
    )
)

# Exibir a tabela com formatação brasileira
display(
    resultado_areas_elevadas.style.format(
        {
            'Área útil': formatar_valor_brasileiro,
            'price': formatar_valor_brasileiro,
            'Preço por m²': formatar_valor_brasileiro,
            'Condomínio': formatar_valor_brasileiro
        }
    )
)


Limite do percentil 99,9%: 1.338,53 m²
Registros acima do percentil 99,9%: 22
Menor área entre os registros investigados: 1.368,00 m²
Maior área entre os registros investigados: 608.608,00 m²


,Área útil,price,Preço por m²,Categoria,Tipo,Quartos,Banheiros,Vagas na garagem,Condomínio,bairro,ZONA
9866,"608.608,00","6.000.000,00","9,86",Casas,Padrão,4,5,5,"5.285,00",Jardim Petrópolis,SUL
8576,"608.608,00","7.000.000,00","11,50",Casas,Padrão,4,5,5,"5.285,00",Jardim Petrópolis,SUL
9861,"608.608,00","6.500.000,00","10,68",Casas,Padrão,4,5,5,"5.285,00",Jardim Petrópolis,SUL
4081,"200.000,00","1.100.000,00","5,50",Casas,Casa de condomínio,4,3,2,"250,00",Barra Funda,CENTRO
10682,"200.000,00","1.100.000,00","5,50",Casas,Casa de condomínio,4,3,2,"250,00",Barra Funda,OESTE
7368,"150.000,00","140.000,00","0,93",Casas,Padrão,2,1,1,"0,00",Jardim Limoeiro,LESTE
3942,"65.120,00","295.000,00","4,53",Casas,Casa de condomínio,2,1,1,"205,00",Vila Rui Barbosa,LESTE
15618,"11.020,00","300.000,00","27,22",Casas,Padrão,2,2,0,"0,00",Parque Císper,LESTE
9182,"3.336,00","255.000,00","76,44",Apartamentos,Padrão,2,1,0,"385,00",Vila Guedes,OESTE
2820,"2.662,00","5.800.000,00","2.178,81",Apartamentos,Padrão,4,5,5,"0,00",Imirim,NORTE


#### Investigação das áreas superiores a 10.000 m²

A análise dos registros acima do percentil **99,9%** identificou **8 imóveis** com Área útil superior a **10.000 m²**.

Esses registros apresentam valores entre **11.020 m² e 608.608 m²**, associados a preços por metro quadrado extremamente baixos, variando aproximadamente entre **R$ 0,93 e R$ 27,22**. Essa relação é incompatível com os demais imóveis da base e reforça a possibilidade de erro de escala, separador decimal ou preenchimento.

Os registros com áreas entre **1.368 m² e 3.336 m²** não serão incluídos neste primeiro tratamento, pois algumas dessas metragens podem ser legítimas para imóveis de grande porte. Esses casos serão investigados separadamente.

Inicialmente, os 8 registros com áreas superiores a 10.000 m² serão comparados com imóveis semelhantes, considerando **Categoria**, **Tipo**, **Quartos**, **Banheiros**, **bairro** e **faixa de preço**. As estimativas e a quantidade de referências disponíveis serão avaliadas antes de qualquer correção ou remoção.

In [ ]:
# Selecionar registros com Área útil superior a 10.000 m²
areas_acima_10mil = df_tratado[
    df_tratado['Área útil'] > 10_000
].copy()

# Criar base de referência sem essas áreas suspeitas
base_referencia_area_alta = df_tratado[
    df_tratado['Área útil'] <= 10_000
].copy()

# Definir faixas de preço pelos percentis da base atual
limites_preco_area = [
    0,
    df_tratado['price'].quantile(0.25),
    df_tratado['price'].quantile(0.50),
    df_tratado['price'].quantile(0.75),
    df_tratado['price'].quantile(0.90),
    float('inf')
]

rotulos_preco_area = [
    'Até Q1',
    'Q1 a Mediana',
    'Mediana a Q3',
    'Q3 a P90',
    'Acima de P90'
]

# Criar as faixas de preço
base_referencia_area_alta['Faixa de preço'] = pd.cut(
    base_referencia_area_alta['price'],
    bins=limites_preco_area,
    labels=rotulos_preco_area,
    include_lowest=True
)

areas_acima_10mil['Faixa de preço'] = pd.cut(
    areas_acima_10mil['price'],
    bins=limites_preco_area,
    labels=rotulos_preco_area,
    include_lowest=True
)

# Calcular estatísticas de imóveis semelhantes
estatisticas_areas_altas = (
    base_referencia_area_alta
    .groupby(
        [
            'Categoria',
            'Tipo',
            'Quartos',
            'Banheiros',
            'Faixa de preço'
        ],
        observed=True
    )['Área útil']
    .agg(
        Quantidade_grupo='count',
        Mediana_sugerida='median'
    )
)

# Associar as estatísticas aos registros suspeitos
analise_areas_acima_10mil = (
    areas_acima_10mil
    .join(
        estatisticas_areas_altas,
        on=[
            'Categoria',
            'Tipo',
            'Quartos',
            'Banheiros',
            'Faixa de preço'
        ]
    )
)

# Calcular o preço por metro quadrado atual
analise_areas_acima_10mil['Preço por m² atual'] = (
    analise_areas_acima_10mil['price'] /
    analise_areas_acima_10mil['Área útil']
).round(2)

# Calcular o preço por metro quadrado com a mediana sugerida
analise_areas_acima_10mil['Preço por m² sugerido'] = (
    analise_areas_acima_10mil['price'] /
    analise_areas_acima_10mil['Mediana_sugerida']
).round(2)

# Organizar o resultado
resultado_areas_acima_10mil = (
    analise_areas_acima_10mil[
        [
            'Área útil',
            'price',
            'Preço por m² atual',
            'Categoria',
            'Tipo',
            'Quartos',
            'Banheiros',
            'bairro',
            'Faixa de preço',
            'Quantidade_grupo',
            'Mediana_sugerida',
            'Preço por m² sugerido'
        ]
    ]
    .sort_values(
        by='Área útil',
        ascending=False
    )
)

# Exibir o resumo
print(f'Registros analisados: {len(resultado_areas_acima_10mil)}')

print(
    f'Registros com mediana disponível: '
    f'{resultado_areas_acima_10mil["Mediana_sugerida"].notna().sum()}'
)

print(
    f'Registros sem referência disponível: '
    f'{resultado_areas_acima_10mil["Mediana_sugerida"].isna().sum()}'
)

# Exibir a tabela formatada
display(
    resultado_areas_acima_10mil.style.format(
        {
            'Área útil': formatar_valor_brasileiro,
            'price': formatar_valor_brasileiro,
            'Preço por m² atual': formatar_valor_brasileiro,
            'Quantidade_grupo': '{:,.0f}',
            'Mediana_sugerida': formatar_valor_brasileiro,
            'Preço por m² sugerido': formatar_valor_brasileiro
        },
        na_rep='Sem referência'
    )
)

Registros analisados: 8
Registros com mediana disponível: 8
Registros sem referência disponível: 0


,Área útil,price,Preço por m² atual,Categoria,Tipo,Quartos,Banheiros,bairro,Faixa de preço,Quantidade_grupo,Mediana_sugerida,Preço por m² sugerido
8576,"608.608,00","7.000.000,00","11,50",Casas,Padrão,4,5,Jardim Petrópolis,Acima de P90,91,"514,00","13.618,68"
9861,"608.608,00","6.500.000,00","10,68",Casas,Padrão,4,5,Jardim Petrópolis,Acima de P90,91,"514,00","12.645,91"
9866,"608.608,00","6.000.000,00","9,86",Casas,Padrão,4,5,Jardim Petrópolis,Acima de P90,91,"514,00","11.673,15"
4081,"200.000,00","1.100.000,00","5,50",Casas,Casa de condomínio,4,3,Barra Funda,Mediana a Q3,1,"200,00","5.500,00"
10682,"200.000,00","1.100.000,00","5,50",Casas,Casa de condomínio,4,3,Barra Funda,Mediana a Q3,1,"200,00","5.500,00"
7368,"150.000,00","140.000,00","0,93",Casas,Padrão,2,1,Jardim Limoeiro,Até Q1,160,"98,00","1.428,57"
3942,"65.120,00","295.000,00","4,53",Casas,Casa de condomínio,2,1,Vila Rui Barbosa,Até Q1,83,"47,00","6.276,60"
15618,"11.020,00","300.000,00","27,22",Casas,Padrão,2,2,Parque Císper,Até Q1,181,"92,00","3.260,87"


#### Tratamento das áreas superiores a 10.000 m²

A investigação identificou **8 registros** com Área útil superior a **10.000 m²**, todos com valores incompatíveis com as características dos imóveis apresentados.

Três registros possuem Área útil igual a **608.608 m²** e características praticamente idênticas. Embora exista uma mediana disponível para esse grupo, a magnitude e a repetição do valor indicam um provável erro grave de preenchimento. Por esse motivo, esses três registros serão removidos.

Outros dois registros possuem Área útil igual a **200.000 m²**. Como a mediana sugerida foi calculada com base em apenas um imóvel semelhante, não existe suporte suficiente para uma correção confiável. Esses registros também serão removidos.

Os três registros restantes, com áreas de **150.000 m²**, **65.120 m²** e **11.020 m²**, possuem medianas apoiadas por grupos com quantidade suficiente de imóveis semelhantes. Esses valores serão substituídos pelas respectivas medianas sugeridas.

Assim, o tratamento será realizado da seguinte forma:

- **3 registros serão corrigidos**;
- **5 registros serão removidos**;
- a base deverá passar de **21.615 para 21.610 registros**;
- após o tratamento, não deverão permanecer áreas superiores a **10.000 m²**.

In [ ]:
# Identificar os registros que serão removidos
indices_remover_areas_altas = resultado_areas_acima_10mil[
    resultado_areas_acima_10mil['Área útil'].isin(
        [608_608, 200_000]
    )
].index

# Identificar os registros que serão corrigidos
indices_corrigir_areas_altas = resultado_areas_acima_10mil[
    resultado_areas_acima_10mil['Área útil'].isin(
        [150_000, 65_120, 11_020]
    )
].index

# Registrar as quantidades antes do tratamento
registros_antes = len(df_tratado)
quantidade_corrigir = len(indices_corrigir_areas_altas)
quantidade_remover = len(indices_remover_areas_altas)

# Substituir as áreas inconsistentes pelas medianas sugeridas
df_tratado.loc[
    indices_corrigir_areas_altas,
    'Área útil'
] = resultado_areas_acima_10mil.loc[
    indices_corrigir_areas_altas,
    'Mediana_sugerida'
].round().astype('int64')

# Remover os registros considerados inválidos
df_tratado.drop(
    index=indices_remover_areas_altas,
    inplace=True
)

# Reorganizar o índice
df_tratado.reset_index(
    drop=True,
    inplace=True
)

# Validar o tratamento
print(f'Registros corrigidos: {quantidade_corrigir}')
print(f'Registros removidos: {quantidade_remover}')
print(f'Registros antes do tratamento: {registros_antes}')
print(f'Registros após o tratamento: {len(df_tratado)}')

print(
    f'Áreas superiores a 10.000 m² após o tratamento: '
    f'{(df_tratado["Área útil"] > 10_000).sum()}'
)

print(
    f'Maior Área útil após o tratamento: '
    f'{df_tratado["Área útil"].max():,.0f} m²'
)

Registros corrigidos: 3
Registros removidos: 5
Registros antes do tratamento: 21615
Registros após o tratamento: 21610
Áreas superiores a 10.000 m² após o tratamento: 0
Maior Área útil após o tratamento: 3,336 m²


#### Investigação das áreas entre 1.000 m² e 10.000 m²

Após o tratamento das áreas claramente incompatíveis, a maior Área útil registrada na base passou a ser **3.336 m²**.

Nesta etapa, serão investigados os imóveis com áreas superiores a **1.000 m²**. Esses registros não serão considerados inválidos automaticamente, pois grandes residências e imóveis de alto padrão podem apresentar metragens elevadas.

A análise considerará o preço por metro quadrado, a Categoria, o Tipo, a localização e as características estruturais de cada imóvel. O objetivo será diferenciar áreas plausíveis de possíveis erros de preenchimento ou escala.

Nenhum registro será corrigido ou removido antes da análise individual dos resultados.

In [ ]:
# Selecionar imóveis com Área útil entre 1.000 m² e 10.000 m²
areas_entre_1000_10000 = df_tratado[
    (df_tratado['Área útil'] > 1_000) &
    (df_tratado['Área útil'] <= 10_000)
].copy()

# Calcular o preço por metro quadrado
areas_entre_1000_10000['Preço por m²'] = (
    areas_entre_1000_10000['price'] /
    areas_entre_1000_10000['Área útil']
).round(2)

# Exibir informações gerais
print(
    f'Registros com Área útil entre 1.000 m² e 10.000 m²: '
    f'{len(areas_entre_1000_10000)}'
)

print(
    f'Menor área investigada: '
    f'{formatar_valor_brasileiro(areas_entre_1000_10000["Área útil"].min())} m²'
)

print(
    f'Maior área investigada: '
    f'{formatar_valor_brasileiro(areas_entre_1000_10000["Área útil"].max())} m²'
)

# Organizar os registros para apresentação
resultado_areas_entre_1000_10000 = (
    areas_entre_1000_10000[
        [
            'Área útil',
            'price',
            'Preço por m²',
            'Categoria',
            'Tipo',
            'Quartos',
            'Banheiros',
            'Vagas na garagem',
            'Condomínio',
            'bairro',
            'ZONA'
        ]
    ]
    .sort_values(
        by='Área útil',
        ascending=False
    )
)

# Exibir a tabela com formatação brasileira
display(
    resultado_areas_entre_1000_10000.style.format(
        {
            'Área útil': formatar_valor_brasileiro,
            'price': formatar_valor_brasileiro,
            'Preço por m²': formatar_valor_brasileiro,
            'Condomínio': formatar_valor_brasileiro
        }
    )
)

Registros com Área útil entre 1.000 m² e 10.000 m²: 30
Menor área investigada: 1.016,00 m²
Maior área investigada: 3.336,00 m²


,Área útil,price,Preço por m²,Categoria,Tipo,Quartos,Banheiros,Vagas na garagem,Condomínio,bairro,ZONA
9180,"3.336,00","255.000,00","76,44",Apartamentos,Padrão,2,1,0,"385,00",Vila Guedes,OESTE
2820,"2.662,00","5.800.000,00","2.178,81",Apartamentos,Padrão,4,5,5,"0,00",Imirim,NORTE
3763,"2.500,00","26.500.000,00","10.600,00",Apartamentos,Padrão,1,1,0,"0,00",Nossa Senhora do Ó,NORTE
8872,"2.400,00","380.494,00","158,54",Apartamentos,Padrão,3,1,1,"0,00",Vila Maria,NORTE
14618,"2.400,00","380.494,00","158,54",Apartamentos,Padrão,3,1,1,"0,00",Vila Maria,NORTE
12309,"2.063,00","211.400,00","102,47",Casas,Padrão,2,1,1,"0,00",Vila Carmosina,LESTE
767,"2.000,00","860.000,00","430,00",Casas,Casa de condomínio,3,3,5,"450,00",Parque Ipê,OESTE
7414,"1.900,00","22.000.000,00","11.578,95",Casas,Casa de condomínio,5,5,5,"0,00",Retiro Morumbi,SUL
347,"1.861,00","5.000.000,00","2.686,73",Apartamentos,Padrão,4,5,5,"0,00",Quarta Parada,LESTE
16716,"1.600,00","18.000.000,00","11.250,00",Casas,Casa de condomínio,4,5,5,"7.300,00",Brooklin Paulista,SUL


#### Comparação das áreas elevadas com imóveis semelhantes

A investigação identificou **30 registros** com Área útil entre **1.000 m² e 10.000 m²**. Parte dessas metragens pode ser legítima, especialmente em casas de alto padrão, imóveis de grande porte ou propriedades localizadas em terrenos extensos.

Entretanto, alguns registros apresentam preços por metro quadrado extremamente baixos ou características incompatíveis com a metragem informada, como apartamentos com áreas superiores a 2.000 m² e preços relativamente reduzidos.

Para diferenciar áreas plausíveis de possíveis erros, os registros serão comparados com imóveis semelhantes, considerando:

- **Categoria**;
- **Tipo**;
- **Quartos**;
- **Banheiros**;
- **faixa de preço**.

A análise apresentará a mediana da Área útil e a quantidade de imóveis disponível em cada grupo. Os registros com Área útil superior a 1.000 m² serão desconsiderados no cálculo das medianas, evitando que os próprios valores investigados influenciem as estimativas.

Nenhum registro será alterado antes da avaliação dos resultados.

In [ ]:
# Criar uma base de referência sem as áreas investigadas
base_referencia_area_elevada = df_tratado[
    df_tratado['Área útil'] <= 1_000
].copy()

# Definir faixas de preço pelos percentis da base atual
limites_preco_area_elevada = [
    0,
    df_tratado['price'].quantile(0.25),
    df_tratado['price'].quantile(0.50),
    df_tratado['price'].quantile(0.75),
    df_tratado['price'].quantile(0.90),
    float('inf')
]

rotulos_preco_area_elevada = [
    'Até Q1',
    'Q1 a Mediana',
    'Mediana a Q3',
    'Q3 a P90',
    'Acima de P90'
]

# Criar as faixas de preço na base de referência
base_referencia_area_elevada['Faixa de preço'] = pd.cut(
    base_referencia_area_elevada['price'],
    bins=limites_preco_area_elevada,
    labels=rotulos_preco_area_elevada,
    include_lowest=True
)

# Criar uma cópia dos 30 registros investigados
analise_areas_elevadas = areas_entre_1000_10000.copy()

analise_areas_elevadas['Faixa de preço'] = pd.cut(
    analise_areas_elevadas['price'],
    bins=limites_preco_area_elevada,
    labels=rotulos_preco_area_elevada,
    include_lowest=True
)

# Calcular a quantidade e a mediana de Área útil por grupo
estatisticas_areas_elevadas = (
    base_referencia_area_elevada
    .groupby(
        [
            'Categoria',
            'Tipo',
            'Quartos',
            'Banheiros',
            'Faixa de preço'
        ],
        observed=True
    )['Área útil']
    .agg(
        Quantidade_grupo='count',
        Mediana_sugerida='median'
    )
)

# Associar as estatísticas aos registros investigados
analise_areas_elevadas = (
    analise_areas_elevadas
    .join(
        estatisticas_areas_elevadas,
        on=[
            'Categoria',
            'Tipo',
            'Quartos',
            'Banheiros',
            'Faixa de preço'
        ]
    )
)

# Calcular o preço por m² com a área atualmente registrada
analise_areas_elevadas['Preço por m² atual'] = (
    analise_areas_elevadas['price'] /
    analise_areas_elevadas['Área útil']
).round(2)

# Calcular o preço por m² com a mediana sugerida
analise_areas_elevadas['Preço por m² sugerido'] = (
    analise_areas_elevadas['price'] /
    analise_areas_elevadas['Mediana_sugerida']
).round(2)

# Organizar a tabela
resultado_areas_elevadas = (
    analise_areas_elevadas[
        [
            'Área útil',
            'price',
            'Preço por m² atual',
            'Categoria',
            'Tipo',
            'Quartos',
            'Banheiros',
            'bairro',
            'Faixa de preço',
            'Quantidade_grupo',
            'Mediana_sugerida',
            'Preço por m² sugerido'
        ]
    ]
    .sort_values(
        by='Área útil',
        ascending=False
    )
)

# Apresentar o resumo
print(f'Registros analisados: {len(resultado_areas_elevadas)}')

print(
    f'Registros com mediana disponível: '
    f'{resultado_areas_elevadas["Mediana_sugerida"].notna().sum()}'
)

print(
    f'Registros sem referência disponível: '
    f'{resultado_areas_elevadas["Mediana_sugerida"].isna().sum()}'
)

print(
    f'Estimativas baseadas em pelo menos 5 imóveis: '
    f'{(resultado_areas_elevadas["Quantidade_grupo"] >= 5).sum()}'
)

# Exibir a tabela com formatação brasileira
display(
    resultado_areas_elevadas.style.format(
        {
            'Área útil': formatar_valor_brasileiro,
            'price': formatar_valor_brasileiro,
            'Preço por m² atual': formatar_valor_brasileiro,
            'Quantidade_grupo': '{:,.0f}',
            'Mediana_sugerida': formatar_valor_brasileiro,
            'Preço por m² sugerido': formatar_valor_brasileiro
        },
        na_rep='Sem referência'
    )
)

Registros analisados: 30
Registros com mediana disponível: 30
Registros sem referência disponível: 0
Estimativas baseadas em pelo menos 5 imóveis: 30


,Área útil,price,Preço por m² atual,Categoria,Tipo,Quartos,Banheiros,bairro,Faixa de preço,Quantidade_grupo,Mediana_sugerida,Preço por m² sugerido
9180,"3.336,00","255.000,00","76,44",Apartamentos,Padrão,2,1,Vila Guedes,Até Q1,"2,269","49,00","5.204,08"
2820,"2.662,00","5.800.000,00","2.178,81",Apartamentos,Padrão,4,5,Imirim,Acima de P90,369,"283,00","20.494,70"
3763,"2.500,00","26.500.000,00","10.600,00",Apartamentos,Padrão,1,1,Nossa Senhora do Ó,Acima de P90,11,"198,00","133.838,38"
8872,"2.400,00","380.494,00","158,54",Apartamentos,Padrão,3,1,Vila Maria,Até Q1,196,"65,00","5.853,75"
14618,"2.400,00","380.494,00","158,54",Apartamentos,Padrão,3,1,Vila Maria,Até Q1,196,"65,00","5.853,75"
12309,"2.063,00","211.400,00","102,47",Casas,Padrão,2,1,Vila Carmosina,Até Q1,160,"98,00","2.157,14"
767,"2.000,00","860.000,00","430,00",Casas,Casa de condomínio,3,3,Parque Ipê,Q1 a Mediana,49,"124,00","6.935,48"
7414,"1.900,00","22.000.000,00","11.578,95",Casas,Casa de condomínio,5,5,Retiro Morumbi,Acima de P90,7,"800,00","27.500,00"
347,"1.861,00","5.000.000,00","2.686,73",Apartamentos,Padrão,4,5,Quarta Parada,Acima de P90,369,"283,00","17.667,84"
16716,"1.600,00","18.000.000,00","11.250,00",Casas,Casa de condomínio,4,5,Brooklin Paulista,Acima de P90,38,"520,00","34.615,38"


#### Seleção dos casos mais suspeitos entre as áreas elevadas

A comparação demonstrou que todos os **30 registros** possuem grupos de referência com pelo menos cinco imóveis. Entretanto, a presença de uma área superior a 1.000 m² não representa, isoladamente, uma inconsistência, pois imóveis de alto padrão podem apresentar metragens elevadas.

Para evitar a correção indevida de áreas legítimas, será utilizado também o **preço por metro quadrado**. Inicialmente, serão selecionados para investigação os registros que apresentam simultaneamente:

- Área útil superior a **1.000 m²**;
- preço por metro quadrado inferior a **R$ 2.500,00**.

Esse critério permitirá concentrar a análise nos imóveis cuja relação entre área e preço apresenta maior incompatibilidade. O limite será utilizado apenas como parâmetro investigativo e nenhum registro será alterado nesta etapa.


In [ ]:
# Selecionar registros com área elevada e preço por m² inferior a R$ 2.500,00
areas_elevadas_suspeitas = resultado_areas_elevadas[
    resultado_areas_elevadas['Preço por m² atual'] < 2_500
].copy()

# Exibir a quantidade de registros identificados
print(
    f'Registros com Área útil superior a 1.000 m² '
    f'e preço por m² inferior a R$ 2.500,00: '
    f'{len(areas_elevadas_suspeitas)}'
)

# Exibir os casos selecionados
display(
    areas_elevadas_suspeitas.style.format(
        {
            'Área útil': formatar_valor_brasileiro,
            'price': formatar_valor_brasileiro,
            'Preço por m² atual': formatar_valor_brasileiro,
            'Quantidade_grupo': '{:,.0f}',
            'Mediana_sugerida': formatar_valor_brasileiro,
            'Preço por m² sugerido': formatar_valor_brasileiro
        },
        na_rep='Sem referência'
    )
)

Registros com Área útil superior a 1.000 m² e preço por m² inferior a R$ 2.500,00: 10


,Área útil,price,Preço por m² atual,Categoria,Tipo,Quartos,Banheiros,bairro,Faixa de preço,Quantidade_grupo,Mediana_sugerida,Preço por m² sugerido
9180,"3.336,00","255.000,00","76,44",Apartamentos,Padrão,2,1,Vila Guedes,Até Q1,"2,269","49,00","5.204,08"
2820,"2.662,00","5.800.000,00","2.178,81",Apartamentos,Padrão,4,5,Imirim,Acima de P90,369,"283,00","20.494,70"
8872,"2.400,00","380.494,00","158,54",Apartamentos,Padrão,3,1,Vila Maria,Até Q1,196,"65,00","5.853,75"
14618,"2.400,00","380.494,00","158,54",Apartamentos,Padrão,3,1,Vila Maria,Até Q1,196,"65,00","5.853,75"
12309,"2.063,00","211.400,00","102,47",Casas,Padrão,2,1,Vila Carmosina,Até Q1,160,"98,00","2.157,14"
767,"2.000,00","860.000,00","430,00",Casas,Casa de condomínio,3,3,Parque Ipê,Q1 a Mediana,49,"124,00","6.935,48"
19663,"1.406,00","3.220.000,00","2.290,18",Casas,Casa de condomínio,5,5,Jardim Ibiratiba,Acima de P90,7,"800,00","4.025,00"
20121,"1.368,00","1.100.000,00","804,09",Apartamentos,Padrão,4,5,Nossa Senhora do Ó,Mediana a Q3,102,"173,00","6.358,38"
12799,"1.139,00","2.200.000,00","1.931,52",Apartamentos,Padrão,3,2,Santa Cecília,Q3 a P90,291,"150,00","14.666,67"
7697,"1.117,00","2.200.000,00","1.969,56",Apartamentos,Padrão,3,2,Vila Nova Parada,Q3 a P90,291,"150,00","14.666,67"


#### Tratamento dos casos suspeitos entre as áreas elevadas

A investigação identificou **10 registros** que apresentam simultaneamente Área útil superior a **1.000 m²** e preço por metro quadrado inferior a **R$ 2.500,00**.

Esses registros possuem relações incompatíveis entre área e preço, indicando possíveis erros de preenchimento ou escala. Todos apresentam grupos de comparação com pelo menos **7 imóveis semelhantes**, oferecendo suporte suficiente para a utilização das medianas sugeridas.

Dessa forma, os 10 valores suspeitos de Área útil serão substituídos pelas respectivas medianas dos grupos formados por **Categoria**, **Tipo**, **Quartos**, **Banheiros** e **faixa de preço**.

Os outros 20 imóveis com áreas superiores a 1.000 m² serão preservados, pois apresentam preços por metro quadrado mais compatíveis com imóveis de grande porte ou alto padrão.

Após a correção, serão verificadas a quantidade de registros tratados, a maior Área útil restante e as estatísticas finais da variável.


In [ ]:
# Identificar os índices dos 10 registros suspeitos selecionados anteriormente
indices_corrigir_areas_suspeitas = areas_elevadas_suspeitas.index

# Registrar a quantidade de registros antes do tratamento
registros_antes = len(df_tratado)
quantidade_corrigir = len(indices_corrigir_areas_suspeitas)

# Substituir somente as áreas suspeitas pelas medianas sugeridas
df_tratado.loc[
    indices_corrigir_areas_suspeitas,
    'Área útil'
] = areas_elevadas_suspeitas.loc[
    indices_corrigir_areas_suspeitas,
    'Mediana_sugerida'
].round().astype('int64')

# Validar o tratamento
print(f'Registros corrigidos: {quantidade_corrigir}')
print('Registros removidos: 0')
print(f'Registros antes do tratamento: {registros_antes}')
print(f'Registros após o tratamento: {len(df_tratado)}')

print(
    f'Áreas superiores a 1.000 m² após o tratamento: '
    f'{(df_tratado["Área útil"] > 1_000).sum()}'
)

print(
    f'Maior Área útil após o tratamento: '
    f'{df_tratado["Área útil"].max():,.0f} m²'
)

print('\nEstatísticas da Área útil após o tratamento:')
print(df_tratado['Área útil'].describe())

Registros corrigidos: 10
Registros removidos: 0
Registros antes do tratamento: 21610
Registros após o tratamento: 21610
Áreas superiores a 1.000 m² após o tratamento: 20
Maior Área útil após o tratamento: 2,500 m²

Estatísticas da Área útil após o tratamento:
count   21,610.00
mean       134.87
std        110.39
min         12.00
25%         65.00
50%        107.00
75%        167.00
max      2,500.00
Name: Área útil, dtype: float64


#### Conclusão do tratamento da variável Área útil

O tratamento da variável **Área útil** foi concluído após a investigação dos valores excessivamente baixos e elevados.

Inicialmente, foram analisados **105 registros** com áreas iguais ou inferiores a **10 m²**. Desse total, **104 registros foram corrigidos** pelas medianas de imóveis semelhantes e **1 registro foi removido** por não possuir suporte suficiente para uma estimativa confiável.

Na análise das áreas extremamente elevadas, foram corrigidos **3 registros** com medianas apoiadas por grupos semelhantes e removidos **5 registros** considerados incompatíveis ou sem referência adequada.

Posteriormente, outros **10 registros** com áreas superiores a 1.000 m² e preços por metro quadrado inferiores a R$ 2.500,00 foram corrigidos. Os demais imóveis de grande porte foram preservados por apresentarem características e valores considerados plausíveis.

Após o tratamento:

- a base permaneceu com **21.610 registros**;
- a menor Área útil passou a ser **12 m²**;
- a maior Área útil passou a ser **2.500 m²**;
- a média foi reduzida para aproximadamente **134,87 m²**;
- o desvio-padrão foi reduzido para aproximadamente **110,39 m²**;
- a variável foi mantida como `float64`, preservando os valores decimais resultantes das estimativas.

Com isso, a variável **Área útil** apresenta uma distribuição mais coerente e adequada para as próximas etapas do projeto.

## 10. Verificação da variável Vagas na garagem

A variável **Vagas na garagem** já foi convertida para o formato numérico, sendo o valor **5** utilizado para representar a categoria original **“5 ou mais”**.

Nesta etapa, será realizada apenas uma verificação dos valores existentes e de sua distribuição. O objetivo é confirmar que a variável contém somente valores válidos entre **0 e 5**, identificar possíveis inconsistências e avaliar se algum tratamento adicional será necessário.

Nenhuma alteração será realizada antes da análise dos resultados.

In [ ]:
# Verificar o tipo da variável
print(
    f'Tipo da variável Vagas na garagem: '
    f'{df_tratado["Vagas na garagem"].dtype}'
)

# Exibir valores existentes e respectivas frequências
print('\nDistribuição dos valores:')

print(
    df_tratado['Vagas na garagem']
    .value_counts()
    .sort_index()
)

# Verificar valores fora do intervalo esperado
vagas_invalidas = (
    (df_tratado['Vagas na garagem'] < 0) |
    (df_tratado['Vagas na garagem'] > 5)
)

print(
    f'\nRegistros com valores fora do intervalo de 0 a 5: '
    f'{vagas_invalidas.sum()}'
)

# Verificar valores ausentes
print(
    f'Valores ausentes: '
    f'{df_tratado["Vagas na garagem"].isna().sum()}'
)

Tipo da variável Vagas na garagem: int64

Distribuição dos valores:
Vagas na garagem
0    2761
1    7460
2    6653
3    2266
4    1416
5    1054
Name: count, dtype: int64

Registros com valores fora do intervalo de 0 a 5: 0
Valores ausentes: 0


### 10.1. Conclusão da verificação da variável Vagas na garagem

A verificação confirmou que a variável **Vagas na garagem** apresenta somente valores entre **0 e 5**, sem valores ausentes ou registros fora do intervalo esperado.

A variável permanece no tipo inteiro (`int64`), sendo o valor **5** utilizado para representar a categoria original **“5 ou mais”**. Como não foram identificadas novas inconsistências, nenhum tratamento adicional foi necessário.

## 11. Verificação das variáveis categóricas

Nesta etapa, serão verificadas conjuntamente as variáveis **Categoria**, **Tipo**, **ZONA**, **location** e **bairro**.

A análise buscará identificar valores ausentes, diferenças de escrita, espaços extras, categorias pouco frequentes e possíveis redundâncias. Somente situações que exigirem tratamento serão investigadas separadamente.

In [ ]:
# Variáveis categóricas mantidas na base
variaveis_categoricas = [
    'Categoria',
    'Tipo',
    'ZONA',
    'location',
    'bairro'
]

# Resumo das variáveis categóricas
resumo_categoricas = pd.DataFrame({
    'Tipo': df_tratado[variaveis_categoricas].dtypes,
    'Valores ausentes': df_tratado[variaveis_categoricas].isna().sum(),
    'Categorias únicas': df_tratado[variaveis_categoricas].nunique(),
    'Valores com espaços extras': [
        (
            df_tratado[coluna].astype(str) !=
            df_tratado[coluna].astype(str).str.strip()
        ).sum()
        for coluna in variaveis_categoricas
    ]
})

display(resumo_categoricas)

# Exibir as categorias de menor cardinalidade
for coluna in ['Categoria', 'Tipo', 'ZONA']:
    print(f'\nDistribuição da variável {coluna}:')
    print(df_tratado[coluna].value_counts(dropna=False))

,Tipo,Valores ausentes,Categorias únicas,Valores com espaços extras
Categoria,object,0,2,0
Tipo,object,0,9,0
ZONA,object,0,5,0
location,object,0,934,0
bairro,object,0,907,0



Distribuição da variável Categoria:
Categoria
Apartamentos    16545
Casas            5065
Name: count, dtype: int64

Distribuição da variável Tipo:
Tipo
Padrão                         20134
Cobertura                        545
Casa de condomínio               492
Loft                             253
Kitnet                           106
Casa de vila                      61
Duplex ou triplex                 16
Venda - casa em rua pública        2
Venda - casa em vila               1
Name: count, dtype: int64

Distribuição da variável ZONA:
ZONA
NORTE     4646
LESTE     4491
OESTE     4379
CENTRO    4300
SUL       3794
Name: count, dtype: int64


### 11.1. Padronização e remoção de redundância

A verificação não identificou valores ausentes ou espaços extras nas variáveis categóricas. As variáveis **Categoria** e **ZONA** também apresentaram categorias consistentes.

Na variável **Tipo**, foram encontradas três categorias relacionadas a casas de vila ou imóveis residenciais comuns:

- **Venda - casa em rua pública**, com 2 registros, será incorporada à categoria **Padrão**;
- **Venda - casa em vila**, com 1 registro, será incorporada à categoria **Casa de vila**.

Essa padronização reduz categorias raras que representam conceitos já existentes na variável.

As variáveis **location** e **bairro** apresentam informações geográficas semelhantes. Como **location** combina a cidade com a localização do imóvel, enquanto **bairro** apresenta diretamente a informação que será utilizada pelo modelo, a variável **location** será removida para evitar redundância.

Após essas alterações, será verificada novamente a distribuição da variável **Tipo** e a permanência das variáveis categóricas na base.

In [ ]:
# Padronizar categorias raras da variável Tipo
df_tratado['Tipo'] = df_tratado['Tipo'].replace({
    'Venda - casa em rua pública': 'Padrão',
    'Venda - casa em vila': 'Casa de vila'
})

# Remover a variável redundante location
df_tratado.drop(
    columns=['location'],
    inplace=True
)

# Validar as alterações
print('Distribuição atualizada da variável Tipo:')
print(df_tratado['Tipo'].value_counts())

print(
    f'\nA variável location permanece na base? '
    f'{"location" in df_tratado.columns}'
)

print(
    f'Dimensões atuais da base: '
    f'{df_tratado.shape}'
)

Distribuição atualizada da variável Tipo:
Tipo
Padrão                20136
Cobertura               545
Casa de condomínio      492
Loft                    253
Kitnet                  106
Casa de vila             62
Duplex ou triplex        16
Name: count, dtype: int64

A variável location permanece na base? False
Dimensões atuais da base: (21610, 29)


## 12. Validação das variáveis binárias

Nesta etapa, será realizada uma verificação conjunta das variáveis binárias relacionadas às características dos imóveis.

O objetivo é confirmar que todas as colunas possuem somente os valores **0** e **1**, não apresentam dados ausentes e estão armazenadas em formato numérico. Somente as variáveis que apresentarem alguma inconsistência exigirão tratamento adicional.

In [ ]:
# Lista das variáveis binárias
variaveis_binarias = [
    'Academia',
    'Elevador',
    'Permitido animais',
    'Piscina',
    'Portaria',
    'Salão de festas',
    'Condomínio fechado',
    'Segurança 24h',
    'Portão eletrônico',
    'Área murada',
    'Área de serviço',
    'Armários na cozinha',
    'Armários no quarto',
    'Churrasqueira',
    'Mobiliado',
    'Quarto de serviço',
    'Ar condicionado',
    'Porteiro 24h',
    'Varanda'
]

# Criar resumo da validação
validacao_binarias = pd.DataFrame({
    'Tipo': df_tratado[variaveis_binarias].dtypes,
    'Valores ausentes': df_tratado[variaveis_binarias].isna().sum(),
    'Quantidade de valores únicos': (
        df_tratado[variaveis_binarias].nunique()
    ),
    'Valores fora de 0 e 1': [
        (~df_tratado[coluna].isin([0, 1])).sum()
        for coluna in variaveis_binarias
    ],
    'Quantidade de 0': [
        (df_tratado[coluna] == 0).sum()
        for coluna in variaveis_binarias
    ],
    'Quantidade de 1': [
        (df_tratado[coluna] == 1).sum()
        for coluna in variaveis_binarias
    ]
})

# Exibir o resultado
display(validacao_binarias)

# Resumo geral
print(
    f'Total de valores ausentes: '
    f'{df_tratado[variaveis_binarias].isna().sum().sum()}'
)

print(
    f'Total de valores fora de 0 e 1: '
    f'{sum((~df_tratado[coluna].isin([0, 1])).sum() for coluna in variaveis_binarias)}'
)

,Tipo,Valores ausentes,Quantidade de valores únicos,Valores fora de 0 e 1,Quantidade de 0,Quantidade de 1
Academia,int64,0,2,0,17093,4517
Elevador,int64,0,2,0,16970,4640
Permitido animais,int64,0,2,0,14226,7384
Piscina,int64,0,2,0,14000,7610
Portaria,int64,0,2,0,20268,1342
Salão de festas,int64,0,2,0,13067,8543
Condomínio fechado,int64,0,2,0,19338,2272
Segurança 24h,int64,0,2,0,20233,1377
Portão eletrônico,int64,0,2,0,21341,269
Área murada,int64,0,2,0,21569,41


Total de valores ausentes: 0
Total de valores fora de 0 e 1: 0


### 12.1. Conclusão da validação das variáveis binárias

A validação confirmou que todas as **19 variáveis binárias** apresentam somente os valores **0 e 1**, não possuem dados ausentes e estão armazenadas no tipo inteiro (`int64`).

Como nenhuma inconsistência foi identificada, não foi necessário realizar alterações nessas variáveis. As colunas foram preservadas para as próximas etapas do projeto.

## 13. Validação final da base tratada

Após a conclusão dos tratamentos, será realizada uma verificação geral da base para confirmar sua consistência e adequação às próximas etapas do projeto.

A validação incluirá:

- dimensões finais da base;
- presença de valores ausentes;
- existência de registros duplicados;
- tipos das variáveis;
- valores mínimos e máximos das variáveis numéricas;
- categorias existentes;
- consistência das variáveis binárias.

Esta etapa não realizará novos tratamentos automaticamente. Qualquer problema identificado será avaliado antes da aplicação de alterações.

In [ ]:
# Validação final resumida da base tratada
print(f'Dimensões finais: {df_tratado.shape[0]} linhas e {df_tratado.shape[1]} colunas')
print(f'Valores ausentes: {df_tratado.isna().sum().sum()}')
print(f'Registros duplicados: {df_tratado.duplicated().sum()}')
print(f'Índice sequencial: {df_tratado.index.equals(pd.RangeIndex(len(df_tratado)))}')

print('\nTipos das variáveis:')
print(df_tratado.dtypes.value_counts())

Dimensões finais: 21610 linhas e 29 colunas
Valores ausentes: 0
Registros duplicados: 1112
Índice sequencial: True

Tipos das variáveis:
int64      24
object      4
float64     1
Name: count, dtype: int64


In [ ]:
# Identificar todas as ocorrências envolvidas em duplicidades
registros_duplicados_finais = df_tratado[
    df_tratado.duplicated(keep=False)
]

# Quantidade de grupos distintos duplicados
grupos_duplicados = (
    registros_duplicados_finais
    .drop_duplicates()
    .shape[0]
)

print(f'Repetições excedentes: {df_tratado.duplicated().sum()}')
print(f'Registros envolvidos em duplicidades: {len(registros_duplicados_finais)}')
print(f'Grupos distintos de registros duplicados: {grupos_duplicados}')


Repetições excedentes: 1112
Registros envolvidos em duplicidades: 1988
Grupos distintos de registros duplicados: 876


### 13.1. Investigação das duplicidades após o tratamento

A validação final identificou **1.112 ocorrências duplicadas excedentes** na base tratada. Ao considerar também as primeiras ocorrências, foram encontrados **1.988 registros distribuídos em 876 grupos de duplicidade**.

Essas duplicidades não serão removidas automaticamente. Após a exclusão de variáveis como **index**, **title** e **date**, anúncios diferentes podem ter passado a apresentar exatamente os mesmos valores nas 29 colunas restantes. Além disso, imóveis distintos podem compartilhar preço, localização e características semelhantes.

Por esse motivo, será investigada a quantidade de ocorrências em cada grupo, com atenção especial às combinações que aparecem muitas vezes. Essa análise permitirá avaliar se os registros representam repetições indevidas ou imóveis diferentes com características coincidentes antes de qualquer decisão de remoçã

In [ ]:
# Contar a quantidade de ocorrências de cada combinação duplicada
frequencia_duplicidades = (
    df_tratado
    .value_counts(dropna=False)
    .reset_index(name='Quantidade de ocorrências')
)

# Manter somente as combinações que aparecem mais de uma vez
frequencia_duplicidades = frequencia_duplicidades[
    frequencia_duplicidades['Quantidade de ocorrências'] > 1
].sort_values(
    by='Quantidade de ocorrências',
    ascending=False
)

# Apresentar um resumo
print(
    f'Maior quantidade de repetições de uma mesma combinação: '
    f'{frequencia_duplicidades["Quantidade de ocorrências"].max()}'
)

print('\nDistribuição da frequência dos grupos duplicados:')

print(
    frequencia_duplicidades['Quantidade de ocorrências']
    .value_counts()
    .sort_index()
)

# Exibir os 10 grupos com mais ocorrências
display(
    frequencia_duplicidades.head(10)
)


Maior quantidade de repetições de uma mesma combinação: 5

Distribuição da frequência dos grupos duplicados:
Quantidade de ocorrências
2    670
3    181
4     20
5      5
Name: count, dtype: int64


,price,Categoria,Tipo,Condomínio,Área útil,Quartos,Banheiros,Vagas na garagem,ZONA,Academia,...,Armários na cozinha,Armários no quarto,Churrasqueira,Mobiliado,Quarto de serviço,Ar condicionado,Porteiro 24h,Varanda,bairro,Quantidade de ocorrências
0,749980,Apartamentos,Cobertura,"1,110.00",84,1,2,1,CENTRO,0,...,0,0,1,0,0,0,0,1,Bela Vista,5
1,4786970,Apartamentos,Padrão,"4,800.00",300,4,3,5,CENTRO,0,...,0,0,0,0,0,0,0,0,Santa Cecília,5
2,216175,Apartamentos,Padrão,0.00,28,1,1,0,CENTRO,0,...,0,0,0,0,0,0,0,0,Liberdade,5
3,4739970,Apartamentos,Padrão,"2,280.00",310,4,5,4,CENTRO,0,...,0,0,0,0,0,0,0,0,Santa Cecília,5
4,2099980,Apartamentos,Cobertura,"2,032.00",220,3,2,2,CENTRO,0,...,0,0,0,0,0,0,0,1,Bela Vista,5
24,1456970,Apartamentos,Cobertura,"1,623.00",83,2,2,2,SUL,0,...,0,0,1,0,0,0,0,0,Indianópolis,4
22,849970,Apartamentos,Padrão,"1,500.00",130,3,2,2,OESTE,0,...,0,0,0,0,0,0,0,0,Alto da Lapa,4
21,1599980,Apartamentos,Padrão,"1,453.00",110,3,3,1,OESTE,0,...,0,0,0,0,0,1,0,0,Pinheiros,4
20,1165970,Apartamentos,Padrão,"1,400.00",110,3,3,1,CENTRO,0,...,0,0,0,0,0,0,0,0,Bela Vista,4
19,995000,Casas,Padrão,0.00,317,4,3,5,CENTRO,0,...,0,0,0,0,0,0,0,0,Cambuci,4


### 13.2. Remoção das duplicidades após o tratamento

A validação final identificou **1.112 ocorrências duplicadas excedentes**, distribuídas em **876 combinações de registros**. Essas duplicidades surgiram após a remoção e a padronização de variáveis, fazendo com que registros anteriormente diferenciados passassem a apresentar valores idênticos nas 29 colunas restantes, incluindo a variável-alvo **price**.

A permanência dessas repetições poderia aumentar artificialmente a influência de determinados padrões e permitir que registros idênticos fossem distribuídos entre os conjuntos de treinamento e teste, produzindo métricas excessivamente otimistas.

Por esse motivo, será mantida apenas a primeira ocorrência de cada combinação, removendo as **1.112 repetições excedentes**. A base deverá passar de **21.610 para 20.498 registros**, quantidade ainda adequada para o treinamento e a avaliação dos futuros modelos de Machine Learning.

In [ ]:
# Registrar a quantidade de registros antes da remoção
registros_antes = len(df_tratado)

# Identificar a quantidade de duplicidades excedentes
duplicados_antes = df_tratado.duplicated().sum()

# Remover duplicidades, mantendo a primeira ocorrência
df_tratado.drop_duplicates(
    keep='first',
    inplace=True
)

# Reorganizar o índice
df_tratado.reset_index(
    drop=True,
    inplace=True
)

# Validar a remoção
print(f'Registros antes da remoção: {registros_antes}')
print(f'Duplicidades removidas: {duplicados_antes}')
print(f'Registros após a remoção: {len(df_tratado)}')
print(f'Duplicidades restantes: {df_tratado.duplicated().sum()}')

Registros antes da remoção: 21610
Duplicidades removidas: 1112
Registros após a remoção: 20498
Duplicidades restantes: 0


### 13.3. Validação final após a remoção das duplicidades

Após a remoção das duplicidades excedentes, será realizada uma última verificação da base tratada.

A validação confirmará as dimensões finais, a ausência de valores faltantes e duplicados, a organização do índice e os tipos de dados. Também será verificado se as principais variáveis permanecem dentro dos limites definidos durante o tratamento.

Essa etapa tem caráter exclusivo de controle de qualidade e não realizará alterações automáticas na base.

In [ ]:
# Validação final da base tratada
print('VALIDAÇÃO FINAL DA BASE TRATADA\n')

print(f'Dimensões finais: {df_tratado.shape[0]} linhas e {df_tratado.shape[1]} colunas')
print(f'Valores ausentes: {df_tratado.isna().sum().sum()}')
print(f'Registros duplicados: {df_tratado.duplicated().sum()}')
print(f'Índice sequencial: {df_tratado.index.equals(pd.RangeIndex(len(df_tratado)))}')

print('\nTipos das variáveis:')
print(df_tratado.dtypes.value_counts())

print('\nLimites das principais variáveis:')
print(f'Menor preço: R$ {formatar_valor_brasileiro(df_tratado["price"].min())}')
print(f'Maior preço: R$ {formatar_valor_brasileiro(df_tratado["price"].max())}')
print(f'Menor Área útil: {formatar_valor_brasileiro(df_tratado["Área útil"].min())} m²')
print(f'Maior Área útil: {formatar_valor_brasileiro(df_tratado["Área útil"].max())} m²')
print(f'Menor Condomínio: R$ {formatar_valor_brasileiro(df_tratado["Condomínio"].min())}')
print(f'Maior Condomínio: R$ {formatar_valor_brasileiro(df_tratado["Condomínio"].max())}')

print('\nIntervalos das variáveis discretas:')
print(f'Quartos: {df_tratado["Quartos"].min()} a {df_tratado["Quartos"].max()}')
print(f'Banheiros: {df_tratado["Banheiros"].min()} a {df_tratado["Banheiros"].max()}')
print(f'Vagas na garagem: {df_tratado["Vagas na garagem"].min()} a {df_tratado["Vagas na garagem"].max()}')


VALIDAÇÃO FINAL DA BASE TRATADA

Dimensões finais: 20498 linhas e 29 colunas
Valores ausentes: 0
Registros duplicados: 0
Índice sequencial: True

Tipos das variáveis:
int64      24
object      4
float64     1
Name: count, dtype: int64

Limites das principais variáveis:
Menor preço: R$ 50.000,00
Maior preço: R$ 45.900.000,00
Menor Área útil: 12,00 m²
Maior Área útil: 2.500,00 m²
Menor Condomínio: R$ 0,00
Maior Condomínio: R$ 25.000,00

Intervalos das variáveis discretas:
Quartos: 0 a 5
Banheiros: 1 a 5
Vagas na garagem: 0 a 5


## 14. Conclusão do tratamento dos dados

O processo de tratamento resultou em uma base mais consistente e adequada para as próximas etapas do projeto de Ciência de Dados.

Entre os principais procedimentos realizados, destacam-se:

- remoção de registros duplicados;
- exclusão de variáveis sem relevância para o objetivo do projeto;
- correção dos tipos de dados;
- tratamento dos valores ausentes;
- correção e remoção de valores inconsistentes em **price**, **Condomínio**, **Área útil**, **Quartos** e **Banheiros**;
- padronização das variáveis categóricas;
- remoção da redundância entre **location** e **bairro**;
- validação das variáveis binárias;
- nova verificação e remoção das duplicidades resultantes das transformações.

Após o tratamento, a base final apresenta:

- **20.498 registros**;
- **29 variáveis**;
- nenhum valor ausente;
- nenhum registro duplicado;
- índice sequencial;
- tipos de dados adequados;
- valores numéricos dentro dos critérios estabelecidos.

A base original foi preservada durante todo o processo. A versão tratada será exportada em um novo arquivo CSV e utilizada posteriormente na preparação e no treinamento dos modelos de Machine Learning.

In [ ]:
# Caminho do novo arquivo CSV
caminho_base_tratada = (
    '/content/drive/MyDrive/Colab Notebooks/PROJETO IV- UNIVESP/'
    'data_preco_sp_tratada.csv'
)

# Exportar a base tratada
df_tratado.to_csv(
    caminho_base_tratada,
    sep=';',
    decimal=',',
    encoding='utf-8-sig',
    index=False
)

# Confirmar a exportação
print('Base tratada exportada com sucesso.')
print(f'Caminho do arquivo: {caminho_base_tratada}')
print(f'Dimensões exportadas: {df_tratado.shape[0]} linhas e {df_tratado.shape[1]} colunas')

Base tratada exportada com sucesso.
Caminho do arquivo: /content/drive/MyDrive/Colab Notebooks/PROJETO IV- UNIVESP/data_preco_sp_tratada.csv
Dimensões exportadas: 20498 linhas e 29 colunas


In [ ]:
# Localizar automaticamente o notebook na pasta do projeto
from pathlib import Path
import subprocess

pasta_projeto = Path(
    '/content/drive/MyDrive/Colab Notebooks/PROJETO IV- UNIVESP'
)

notebooks = list(
    pasta_projeto.glob('*TRATAMENTO_DADOS.ipynb')
)

if not notebooks:
    raise FileNotFoundError('Notebook de tratamento não encontrado.')

caminho_notebook = notebooks[0]
nome_html = 'PRECO_IMOVEIS_SP_TRATAMENTO_DADOS.html'

# Converter o notebook completo para HTML
subprocess.run(
    [
        'jupyter',
        'nbconvert',
        '--to', 'html',
        '--template', 'lab',
        '--output', nome_html,
        '--output-dir', str(pasta_projeto),
        str(caminho_notebook)
    ],
    check=True
)

caminho_html = pasta_projeto / nome_html

print('Arquivo HTML gerado com sucesso.')
print(f'Caminho: {caminho_html}')

Arquivo HTML gerado com sucesso.
Caminho: /content/drive/MyDrive/Colab Notebooks/PROJETO IV- UNIVESP/PRECO_IMOVEIS_SP_TRATAMENTO_DADOS.html
